<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/Phase2_LearnedDistance_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Learned Distances for HRP

* Research question: Do learned asset distances improve HRP portfolios relative to correlation and PCA distances?

* The experiments fit distances using each training window, then evaluate the following test window. Read pooled portfolio performance separately from mean window-level comparisons.


## Notebook map

1. Setup and evaluation windows
2. Shared HRP allocator and metrics
3. Correlation benchmark
4. Autoencoder distance (AE-HRP)
5. Feature-based metric learning (SML-HRP)
6. Comparisons with PCA
7. Shrinkage toward PCA allocations
8. SML with an ERC objective (SML-ERC-HRP)
9. CMA-ES optimizer comparison
10. HERC-style allocator
11. ERC weighting of autoencoder embeddings
12. Robustness checks
13. Weight and concentration diagnostics
14. Cross-method summaries
15. Consolidated inference and daily costs
16. Deflated Sharpe Ratio


## 1. Setup and evaluation windows

* Use the same ten-stock universe, 252-observation training windows and 63-observation test windows as Phase 1. Training windows overlap; test windows do not.

* The download has no fixed end date. Keeping the first 79 windows limits the reference period but does not freeze historical adjusted prices. Individual models use explicit seeds in addition to the global NumPy seed.


In [ ]:
# Install the packages used for data download and the main distance models.

# Run once per fresh runtime. --quiet reduces installation output.
!pip install yfinance scikit-learn --quiet

In [ ]:
# Import shared libraries. Individual estimators also receive explicit seeds.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.neural_network import MLPRegressor
import yfinance as yf

# Seed NumPy’s legacy global generator. Functions using default_rng or estimator
# random_state have their own random streams and receive separate seeds below.
np.random.seed(0)

In [ ]:
# Define the ten-stock universe and keep a consistent column order.

TICKERS = {
    'Energy': 'XOM', 'Materials': 'LIN', 'Industrials': 'HON',
    'Consumer Discretionary': 'AMZN', 'Consumer Staples': 'PG',
    'Healthcare': 'JNJ', 'Financials': 'JPM',
    'Information Technology': 'MSFT', 'Communication Services': 'GOOGL',
    'Utilities': 'NEE',
}
# Dictionary insertion order defines the common asset order used throughout.
# N is the number of stocks, not the number of return observations.
SYMBOLS = list(TICKERS.values())
N = len(SYMBOLS)
print(f"Universe: {SYMBOLS}")

In [ ]:
# Download adjusted prices, forward-fill gaps, and calculate complete daily returns.
# The download has no fixed end date; freeze the data separately for reproducibility.

# auto_adjust=True requests adjusted prices. Each column represents one ticker.
raw = yf.download(SYMBOLS, start='2004-01-01', auto_adjust=True)['Close']
raw = raw[SYMBOLS]
# Carry the last observed price forward; an internal filled day produces a zero
# return. Leading gaps remain missing until the first price is observed.
prices = raw.ffill()
# Compute simple returns P[t]/P[t-1] - 1. dropna removes any row missing an asset,
# so every retained date has a complete cross-section in the same column order.
returns = prices.pct_change().dropna()
print(f"Returns shape: {returns.shape}")
assert returns.isnull().sum().sum() == 0, "NaN values found"

In [ ]:
# Store inclusive training and test boundaries. Move forward by one test period.
# Training windows overlap; test windows do not.

TRAIN_DAYS = 252
TEST_DAYS  = 63
FIRST_TRAIN_START = '2005-08-19'

def build_windows(returns, train_days, test_days, first_train_start):
    idx = returns.index
    # Find the first available observation on or after the requested start date.
    # Window lengths count observations, rather than calendar days.
    start = idx.searchsorted(pd.Timestamp(first_train_start))
    windows, pos = [], start
    # Only emit windows with a complete training block and following test block.
    # Tuple order is (train start, train end, test start, test end); endpoints are inclusive.
    while pos + train_days + test_days <= len(idx):
        windows.append((idx[pos], idx[pos+train_days-1],
                        idx[pos+train_days], idx[pos+train_days+test_days-1]))
        # Advance one test block. Consecutive training samples overlap by 189 rows
        # with the default settings, while their 63-row test samples are disjoint.
        pos += test_days
    return windows

windows = build_windows(returns, TRAIN_DAYS, TEST_DAYS, FIRST_TRAIN_START)
print(f"Total windows: {len(windows)}")
print(f"First: {windows[0][0].date()} - {windows[0][3].date()}")
print(f"Last:  {windows[-1][0].date()} - {windows[-1][3].date()}")

In [ ]:
# Keep at most the first 79 complete windows from this downloaded dataset.
# The reported start date is a training date; this does not freeze historical prices.

# Slicing retains fewer than 79 windows if the download does not supply enough.
# Later labels that print 79 are historical assumptions, not checks of this count.
windows = windows[:79]
print(f"Truncated to fixed 79-window reference period: {len(windows)} windows")
print(f"First: {windows[0][0].date()} - {windows[0][3].date()}")
print(f"Last:  {windows[-1][0].date()} - {windows[-1][3].date()}")

## 2. Shared HRP allocator

* Single-linkage clustering supplies an asset order.
* Recursive bisection repeatedly halves that order and allocates more weight to the side with lower estimated cluster variance.
* Cluster risk uses inverse-variance weights and raw training-return covariance.


In [ ]:
def quasi_diag(link, num_items):
    # Linkage columns 0 and 1 hold child IDs; IDs below the asset count are leaves.
    # Larger IDs refer to earlier linkage rows and must be expanded to find the assets.
    link = link.astype(int)
    # Start from the two children of the final merge, which covers all assets.
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    num_items = link[-1, 3]
    while sort_ix.max() >= num_items:
        # Leave alternate index positions free so each cluster can be replaced by
        # its left and right children without losing their traversal order.
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        df0 = sort_ix[sort_ix >= num_items]
        i = df0.index
        # Convert internal cluster IDs to zero-based linkage-row positions.
        j = df0.values - num_items
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sort_ix = pd.concat([sort_ix, df0]).sort_index()
        sort_ix.index = range(sort_ix.shape[0])
    return sort_ix.tolist()

def cluster_var(cov_slice):
    # Within this cluster, use inverse individual variance as provisional weights.
    # This assumes strictly positive diagonal variances.
    ivp = 1.0 / np.diag(cov_slice.values)
    ivp /= ivp.sum()
    # Evaluate the provisional portfolio using the entire cluster covariance,
    # including cross-asset covariances; this scalar drives the next allocation split.
    return ivp @ cov_slice.values @ ivp

def get_rec_bipart(cov, sort_ix):
    # Start every asset at weight 1; successive splits multiply its allocation
    # by the fraction assigned to its side of the ordered asset list.
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix]
    while len(c_items) > 0:
        # Split each nonsingleton group at its midpoint. These are halves of the
        # leaf order, not necessarily the two child clusters at each dendrogram merge.
        c_items = [i[j:k] for i in c_items
                   for j, k in ((0, len(i) // 2), (len(i) // 2, len(i)))
                   if len(i) > 1]
        for i in range(0, len(c_items), 2):
            c0, c1 = c_items[i], c_items[i + 1]
            w0 = cluster_var(cov.loc[c0, c0])
            w1 = cluster_var(cov.loc[c1, c1])
            # Left allocation = right variance / sum of both variances.
            # The lower-risk side receives the larger share; cross-side covariance is omitted.
            alpha = 1 - w0 / (w0 + w1)
            w[c0] *= alpha
            w[c1] *= 1 - alpha
    return w

def hrp_weights_from_dist(returns_df, dist_df):
    cov = returns_df.cov()
    # Convert the square distance matrix to its upper-triangle vector for linkage.
    # checks=False skips symmetry/diagonal validation, so input validity is assumed.
    condensed = squareform(dist_df.values, checks=False)
    link = linkage(condensed, method='single')
    sort_ix_pos = quasi_diag(link, len(returns_df.columns))
    # Map integer leaf positions back to ticker labels before covariance slicing.
    sort_ix = [returns_df.columns[i] for i in sort_ix_pos]
    w = get_rec_bipart(cov, sort_ix)
    # Normalize to full investment, then restore the shared SYMBOLS output order.
    return (w / w.sum()).reindex(SYMBOLS)

def get_corr_distance(returns_df):
    corr = returns_df.corr()
    # Transform correlation +1 to distance 0 and correlation -1 to distance 1.
    return np.sqrt((1 - corr) / 2)

print("HRP harness defined.")

### 2.1 Metrics and paired comparisons
Sharpe uses the five-lag Bartlett-weighted serial-correlation adjustment. Paired inference uses circular blocks of four consecutive windows, a null-centred two-sided test and a 95% basic bootstrap interval. Block-length sensitivity follows at the end. Regime samples are small and results remain exploratory.


In [ ]:
# Define pooled metrics and a paired bootstrap of mean window Sharpe differences.
# Circular blocks preserve local window dependence; inference remains exploratory.

def sharpe_lo2002(returns_series, periods=252, n_lags=5):
    """Autocorrelation-adjusted Sharpe ratio (Lo, 2002)."""
    # Use the sample mean and sample variance of daily simple returns.
    # No risk-free series is subtracted; the numerator is the raw mean return.
    mu, var = returns_series.mean(), returns_series.var()
    adj = 0.0
    for k in range(1, n_lags + 1):
        # Estimate serial correlation at lag k and taper its contribution with a
        # Bartlett weight. Missing autocorrelation estimates are omitted.
        rho = returns_series.autocorr(lag=k)
        if not np.isnan(rho):
            adj += (1 - k / (n_lags + 1)) * rho
    # Inflate or reduce daily variance using the estimated serial correlations.
    # The floor prevents a zero/negative denominator but can produce extreme ratios;
    # this is the implemented five-lag adjustment, not a full 252-lag annual formula.
    return (mu / np.sqrt(max(var * (1 + 2 * adj), 1e-12))) * np.sqrt(periods)

def max_drawdown(r):
    # Corrected 2026-09-09: initial wealth of 1 is now prepended to the wealth
    # path, so a loss on the very first observation is no longer excluded from
    # the running peak used to compute drawdown.
    cum = pd.concat([pd.Series([1.0]), (1 + r).cumprod()], ignore_index=True)
    peak = cum.cummax()
    return ((cum - peak) / peak).min()

def ceq(r, gamma, periods=252):
    # gamma controls risk aversion: larger gamma increases the variance penalty.
    # This is annualized mean-variance utility, rather than compounded annual return.
    return r.mean() * periods - (gamma / 2) * r.var() * periods

def quarterly_sharpes(chunks):
    return np.array([sharpe_lo2002(c) for c in chunks])

def bootstrap_sharpe_diff(chunks_a, chunks_b, n_boot=5000, seed=42, block_length=4):
    """Circular block bootstrap of paired window Sharpes; null-centred two-sided test.

    Default block length is four quarterly windows, with sensitivity reported below.
    This preserves local dependence within sampled blocks, not all dependence.
    Short regime samples remain exploratory. CI is a basic bootstrap interval.
    """
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = quarterly_sharpes(chunks_a) - quarterly_sharpes(chunks_b)
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window Sharpe; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)

def m(chunks): return quarterly_sharpes(chunks).mean()  # per-window mean Sharpe; shared helper for cross-method comparisons

print("Metrics and significance test defined.")

In [ ]:
# Apply equal weights to each day in the test windows. Returns exclude trading costs.
# The inverse squared-weight sum measures effective holdings, not independent bets.

# The matrix product produces one portfolio return for each test day. Applying
# the same weights daily implies constant-mix rebalancing, rather than drifting holdings.
ew_rets = [(returns.loc[vs:ve] @ pd.Series(1.0/N, index=SYMBOLS)).rename('EqualWeight') for (ts,te,vs,ve) in windows]
ew_series = pd.concat(ew_rets)
# For fully invested equal weights, inverse concentration equals N exactly.
enb_ew = 1.0 / np.sum((np.ones(N)/N)**2)
print("=== 1/N (equal-weight) benchmark, 79 walk-forward windows ===")
print(f"Sharpe (Lo 2002): {sharpe_lo2002(ew_series):.4f}")
print(f"Max Drawdown: {max_drawdown(ew_series):.4f}")
print(f"CEQ (gamma=1): {ceq(ew_series, 1):.4f}")
print(f"CEQ (gamma=3): {ceq(ew_series, 3):.4f}")
print(f"Effective Number of Bets (ENB): {enb_ew:.4f}")
print(f"N windows: {len(ew_rets)}")

## 3. Correlation benchmark

* Fit correlation-HRP on training returns and evaluate the following test window.
* The first training date is in 2005; test performance begins after 252 training observations, in 2006.
* Fixed daily target weights are applied within each test window; these returns exclude trading costs.


In [ ]:
# Fit correlation distances and HRP allocations using training data only.
# Apply constant target weights to test days; record quarterly target-weight changes.

base_rets, base_chunks, base_turnover = [], [], []
prev_w = None
for ts, te, vs, ve in windows:
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    dist = get_corr_distance(tr)
    w = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ w[SYMBOLS]).rename('CorrHRP')
    base_rets.append(r); base_chunks.append(r)
    if prev_w is not None:
        # Record the full L1 change between consecutive target allocations.
        # This excludes initial entry, weight drift and trading to maintain targets within a window.
        base_turnover.append(np.abs(w - prev_w).sum())
    prev_w = w

base_series = pd.concat(base_rets)
print("=== Baseline: Correlation-distance HRP (champion proxy) ===")
print(f"Sharpe (Lo 2002):     {sharpe_lo2002(base_series):.3f}")
print(f"Max Drawdown:         {max_drawdown(base_series):.4f}")
print(f"CEQ (gamma=1):        {ceq(base_series, 1):.4f}")
print(f"CEQ (gamma=3):        {ceq(base_series, 3):.4f}")
print(f"Avg quarterly turnover: {np.mean(base_turnover):.4f}")

## 4. Autoencoder distance

* Pool overlapping 20-day standardized return sequences across assets.

* Train a 20–8–20 reconstruction network, then embed each asset using its last 20 training days.

* Euclidean distances between the eight-dimensional embeddings feed HRP.

* Outer test returns are excluded from fitting.


In [ ]:
# Pool overlapping standardized training sequences and fit a reconstruction network.
# Embed each asset with its last 20 training days; compare hidden activations.

def build_ae_training_matrix(train_returns, L=20):
    X = []
    for col in train_returns.columns:
        s = train_returns[col].values
        # Standardize each asset using its entire outer training window.
        # The small offset prevents division by zero for constant return sequences.
        mu, sd = s.mean(), s.std() + 1e-12
        s_std = (s - mu) / sd
        # Create len(s)-L+1 overlapping samples per asset; each has L time coordinates.
        # Samples from different assets are pooled for one reconstruction network.
        for i in range(len(s_std) - L + 1):
            X.append(s_std[i:i + L])
    return np.array(X)

def fit_autoencoder(X, bottleneck_dim=8, seed=0):
    # Use one tanh hidden layer of bottleneck_dim units and a regression output.
    # At the defaults, the network maps 20 inputs to 8 hidden units to 20 outputs.
    model = MLPRegressor(hidden_layer_sizes=(bottleneck_dim,), activation='tanh',
                          solver='adam', max_iter=500, random_state=seed,
                          early_stopping=True, n_iter_no_change=15)
    # The target is the input itself: learn reconstruction rather than future returns.
    # Early stopping holds out an internal subset; overlapping samples can share dates.
    model.fit(X, X)
    return model

def encode(model, X):
    # Extract first-layer activations directly. The decoder/output layer is not
    # used when computing the asset embeddings.
    return np.tanh(X @ model.coefs_[0] + model.intercepts_[0])

def get_embedding_distance(train_returns, L=20, bottleneck_dim=8, seed=0):
    """Fits the autoencoder on TRAIN-window data only, embeds each asset's own
    last L-day TRAIN sub-sequence, returns the pairwise embedding distance matrix.
    No TEST-window data is used anywhere in this function."""
    X_ae = build_ae_training_matrix(train_returns, L=L)
    model = fit_autoencoder(X_ae, bottleneck_dim=bottleneck_dim, seed=seed)
    embs = {}
    for col in train_returns.columns:
        s = train_returns[col].values
        mu, sd = s.mean(), s.std() + 1e-12
        # Use the final L training observations as this asset’s representation,
        # scaled by the same full-training mean and standard deviation used for fitting.
        last_window = ((s[-L:] - mu) / sd).reshape(1, -1)
        embs[col] = encode(model, last_window)[0]
    # Rows are assets; columns are latent coordinates. Pairwise Euclidean norms
    # produce an asset-by-asset distance matrix aligned to train_returns.columns.
    E = pd.DataFrame(embs).T
    D = pd.DataFrame(index=train_returns.columns, columns=train_returns.columns, dtype=float)
    for i in train_returns.columns:
        for j in train_returns.columns:
            D.loc[i, j] = np.linalg.norm(E.loc[i] - E.loc[j])
    return D

print("Autoencoder embedding distance defined.")

In [ ]:
# Refit the autoencoder for each window and evaluate HRP on the next test period.
# Inspect convergence and interruption warnings before treating a run as complete.

ae_rets, ae_chunks, ae_turnover = [], [], []
prev_w_ae = None
#walk forward loop
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    dist = get_embedding_distance(tr, L=20, bottleneck_dim=8, seed=k)  # refit every window, TRAIN-only
    w = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ w[SYMBOLS]).rename('AE-HRP')
    ae_rets.append(r); ae_chunks.append(r)
    if prev_w_ae is not None:
        # Record the full L1 change between consecutive target allocations.
        # This excludes initial entry, weight drift and trading to maintain targets within a window.
        ae_turnover.append(np.abs(w - prev_w_ae).sum())
    prev_w_ae = w

ae_series = pd.concat(ae_rets)
print("=== Experiment 1: Autoencoder-embedding-distance HRP ===")
print(f"Sharpe (Lo 2002):     {sharpe_lo2002(ae_series):.3f}")
print(f"Max Drawdown:         {max_drawdown(ae_series):.4f}")
print(f"CEQ (gamma=1):        {ceq(ae_series, 1):.4f}")
print(f"CEQ (gamma=3):        {ceq(ae_series, 3):.4f}")
print(f"Avg quarterly turnover: {np.mean(ae_turnover):.4f}")

### 4.1 Autoencoder versus correlation

Compare paired window Sharpe differences and display pooled portfolio metrics. The bootstrap difference need not equal the difference between the pooled Sharpe values. PCA comparisons follow in Section 6.


In [ ]:
# The reported delta and interval summarize paired per-window differences.
# The table below instead recomputes metrics on concatenated daily returns.
# Neither the raw bootstrap tail value nor this cell adjusts for all model searches.

# Compare average paired window Sharpes, then display pooled performance.
# The two Sharpe summaries need not have the same difference.

obs, lo, hi, p = bootstrap_sharpe_diff(ae_chunks, base_chunks)
print("=== AE-HRP vs Correlation-HRP (champion proxy) ===")
print(f"Delta Sharpe:  {obs:+.4f}")
print(f"95% CI:        [{lo:+.4f}, {hi:+.4f}]")
print(f"p-value:       {p:.4f}")
print()
print("Summary table")
print("-" * 60)
summary = pd.DataFrame({
    'Sharpe':          [sharpe_lo2002(base_series), sharpe_lo2002(ae_series)],
    'MaxDD':           [max_drawdown(base_series),   max_drawdown(ae_series)],
    'CEQ (g=1)':       [ceq(base_series, 1),         ceq(ae_series, 1)],
    'CEQ (g=3)':       [ceq(base_series, 3),         ceq(ae_series, 3)],
    'Avg Turnover':    [np.mean(base_turnover),       np.mean(ae_turnover)],
}, index=['Correlation-HRP (champion proxy)', 'AE-Embedding-HRP (learned)'])
print(summary.round(4).to_string())

In [ ]:
# cumprod compounds each day’s simple return. The first plotted point is
# after the first return; an explicit initial wealth point of 1 is not prepended.

# Plot gross compounded return growth from the aligned test windows.

fig, ax = plt.subplots(figsize=(10, 5))
(1 + base_series).cumprod().plot(ax=ax, label='Correlation-HRP (champion proxy)')
(1 + ae_series).cumprod().plot(ax=ax, label='AE-Embedding-HRP (learned)')
ax.set_title('Out-of-sample cumulative growth: fixed vs. learned distance')
ax.set_ylabel('Cumulative growth of $1')
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Calendar-regime comparison

Assign windows by test start date: Pre-GFC before 2008, GFC during 2008–2009, Post-GFC-to-COVID during 2010–2019, and COVID+ from 2020 onward. A window crossing a boundary retains its start-date label. These are calendar buckets, not estimated stress states.


In [ ]:
# Select the same calendar-regime windows for both candidates.

regime_bounds = [
    ('Pre-GFC',           pd.Timestamp('1900-01-01'), pd.Timestamp('2008-01-01')),
    ('GFC',               pd.Timestamp('2008-01-01'), pd.Timestamp('2010-01-01')),
    ('Post-GFC-to-COVID', pd.Timestamp('2010-01-01'), pd.Timestamp('2020-01-01')),
    ('COVID+',            pd.Timestamp('2020-01-01'), pd.Timestamp('2100-01-01')),
]
def regime_of(test_start):
    for name, lo, hi in regime_bounds:
        if lo <= test_start < hi:
            return name
    return 'Unknown'

regime_labels = [regime_of(vs) for (ts, te, vs, ve) in windows]

rows = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    base_c = [base_chunks[i] for i in idx]
    ae_c   = [ae_chunks[i] for i in idx]
    base_r = pd.concat(base_c)
    ae_r   = pd.concat(ae_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(ae_c, base_c)
    rows.append({
        'Regime': name,
        'N windows': len(idx),
        'Corr-HRP Sharpe': sharpe_lo2002(base_r),
        'AE-HRP Sharpe': sharpe_lo2002(ae_r),
        'Delta Sharpe': obs,
        '95% CI lo': lo_ci,
        '95% CI hi': hi_ci,
        'p-value': p,
    })

regime_table = pd.DataFrame(rows).set_index('Regime')
print("=== Regime sub-split: AE-Embedding-HRP vs Correlation-HRP ===")
print(regime_table.round(4).to_string())

## 5. Feature-based metric learning (SML-HRP)

* Build six training-period asset statistics and standardize each feature across assets.
* Fit nonnegative feature-distance weights to reduce the resulting HRP portfolio variance.
* This is portfolio-objective-driven fitting, without explicit similarity labels or return-prediction targets.

* Nelder-Mead is derivative-free.


In [ ]:
# Build and standardize six training features across assets.
# Square optimizer parameters to obtain nonnegative distance weights.
# Keep the lowest variance found across restarts; convergence is not checked here.

from scipy.optimize import minimize

FEATURE_NAMES = ['mean_ret', 'vol', 'skew', 'kurt', 'avg_corr', 'downside_dev']

def build_asset_features(train_returns):
    #computing full 10x10 pairwise correlation matrix over TRAIN window once, upfront
    #reused below for every asset's avg corr
    corr = train_returns.corr()
    feats = {}
    #loop over each of the 10 assets
    for col in train_returns.columns:
        #indiv asset's train period daily return series
        s = train_returns[col]
        #daily mean return annualised by multiplying by 252 trading days/yr
        mean_ret = s.mean() * 252
        #daily volatility annualised by standard sqrt(252) scaling
        vol = s.std() * np.sqrt(252)
        # Skewness measures asymmetry; pandas kurtosis is excess kurtosis
        skew = s.skew()
        kurt = s.kurt()
        # Mean correlation with other assets, excluding self-correlation.
        avg_corr = (corr[col].sum() - 1) / (len(corr.columns) - 1)
        #subset of only negative-return days
        downside = s[s < 0]
        # Annualized sample standard deviation among negative-return days.
        downside_dev = downside.std() * np.sqrt(252) if len(downside) > 1 else 0.0
        feats[col] = [mean_ret, vol, skew, kurt, avg_corr, downside_dev]
    # Transpose to an assets-by-six-features table. Standardizing down each column
    # puts features on comparable cross-sectional scales before distance weighting.
    F = pd.DataFrame(feats, index=FEATURE_NAMES).T
    # Standardize each feature across assets in this training window.
    F = (F - F.mean()) / (F.std() + 1e-12)
    return F.copy()

def weighted_feature_distance(F, w):
    w = np.asarray(w)
    #pulls raw array out of dataframe and preallocates the 10*10 output distance matrix
    Fv = F.values
    n = len(Fv)
    Dm = np.zeros((n, n))
    for i in range(n):
        # For asset i, compute squared coordinate differences from every asset.
        # The weighted sum followed by a square root is a diagonal weighted Euclidean distance.
        diff = (Fv - Fv[i])**2
        Dm[i] = np.sqrt((diff * w).sum(axis=1))
    np.fill_diagonal(Dm, 0.0)
    return pd.DataFrame(Dm, index=F.index, columns=F.index)

def insample_variance_objective(theta, F, train_returns):
    # Optimize unrestricted theta, but use theta squared as nonnegative feature weights.
    # These are distance-coordinate weights, not portfolio holdings.
    w = theta**2
    if w.sum() < 1e-8:
        return 1e6
    # Rebuild the distance and resulting HRP allocation at every objective evaluation.
    # Clustering changes discretely; the objective can be flat between ordering changes.
    dist = weighted_feature_distance(F, w)
    try:
        wgt = hrp_weights_from_dist(train_returns, dist)
    except Exception:
        return 1e6
    cov = train_returns.cov()
    # Evaluate daily in-sample portfolio variance w_portfolio.T @ covariance @ w_portfolio.
    # NumPy multiplication is positional: covariance and holdings must share asset order.
    var = wgt.values @ cov.values @ wgt.values
    return float(var)

#computing 6 standardised features once per window
def fit_metric_weights(train_returns, n_restarts=4, maxiter=60, seed=0):

    feats_df = build_asset_features(train_returns)

    rng = np.random.default_rng(seed)
    best = None

    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=len(FEATURE_NAMES))
        # Run derivative-free Nelder-Mead from each random starting point. maxiter limits
        # iterations per restart; the code keeps the lowest reported objective without
        # checking res.success, so a retained fit need not have converged.
        res = minimize(insample_variance_objective, theta0, args=(feats_df, train_returns),
                        method='Nelder-Mead',
                        options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    if best is None:
        # Fallback gives equal distance-feature weights; HRP still determines holdings.
        return feats_df, np.ones(len(FEATURE_NAMES))
    return feats_df, best[1]

print('Supervised metric learning functions defined.')

In [ ]:
# Fit feature-distance weights, build HRP allocations and evaluate the next window.
# Log feature weights separately from portfolio allocations for later diagnostics.

sml_rets, sml_chunks, sml_turnover, learned_weights_log = [], [], [], []
prev_w_sml = None
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    feats_df, w = fit_metric_weights(tr, n_restarts=4, maxiter=60, seed=k)
    dist = weighted_feature_distance(feats_df, w)
    wgt = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    r = (tst @ wgt[SYMBOLS]).rename('SML-HRP')
    sml_rets.append(r); sml_chunks.append(r)
    learned_weights_log.append(w)
    if prev_w_sml is not None:
        # Record the full absolute change between consecutive target allocations.
        sml_turnover.append(np.abs(wgt - prev_w_sml).sum())
    prev_w_sml = wgt

sml_series = pd.concat(sml_rets)
print("=== Experiment 2: Supervised-metric-learning-distance HRP ===")
print(f"Sharpe (Lo 2002):     {sharpe_lo2002(sml_series):.3f}")
print(f"Max Drawdown:         {max_drawdown(sml_series):.4f}")
print(f"CEQ (gamma=1):        {ceq(sml_series, 1):.4f}")
print(f"CEQ (gamma=3):        {ceq(sml_series, 3):.4f}")
print(f"Avg quarterly turnover: {np.mean(sml_turnover):.4f}")

In [ ]:
# Feature-weight logs describe the fitted distance. Portfolio metrics describe
# the HRP holdings obtained from that distance, a different set of weights.

# Report SML versus correlation and pooled portfolio metrics.
# Normalize feature weights per window before interpreting relative importance.

obs, lo, hi, p = bootstrap_sharpe_diff(sml_chunks, base_chunks)
print("=== SML-HRP vs Correlation-HRP (champion proxy) ===")
print(f"Delta Sharpe:  {obs:+.4f}")
print(f"95% CI:        [{lo:+.4f}, {hi:+.4f}]")
print(f"p-value:       {p:.4f}")
print()
# These printed means are RAW weights. Multiplying all feature weights by the
# same positive constant rescales distances without changing the HRP ordering,
# so raw means are not normalized feature-importance estimates.
avg_w = np.mean(learned_weights_log, axis=0)
print("Avg learned feature weights across windows:")
for name, val in zip(FEATURE_NAMES, avg_w):
    print(f"  {name}: {val:.3f}")

print()
summary2 = pd.DataFrame({
    'Sharpe':       [sharpe_lo2002(base_series), sharpe_lo2002(ae_series), sharpe_lo2002(sml_series)],
    'MaxDD':        [max_drawdown(base_series), max_drawdown(ae_series), max_drawdown(sml_series)],
    'CEQ (g=1)':    [ceq(base_series, 1), ceq(ae_series, 1), ceq(sml_series, 1)],
    'CEQ (g=3)':    [ceq(base_series, 3), ceq(ae_series, 3), ceq(sml_series, 3)],
    'Avg Turnover': [np.mean(base_turnover), np.mean(ae_turnover), np.mean(sml_turnover)],
}, index=['Correlation-HRP (champion proxy)', 'AE-Embedding-HRP', 'SML-HRP (Experiment 2)'])
print(summary2.round(4).to_string())

In [ ]:
# The lines share the same test-window schedule if all preceding loops completed.
# No cost adjustment or confidence band is included in this growth chart.

# Compare gross cumulative growth for correlation, AE and SML.

fig, ax = plt.subplots(figsize=(10, 5))
(1 + base_series).cumprod().plot(ax=ax, label='Correlation-HRP (champion proxy)')
(1 + ae_series).cumprod().plot(ax=ax, label='AE-Embedding-HRP')
(1 + sml_series).cumprod().plot(ax=ax, label='SML-HRP (Experiment 2)')
ax.set_title('Out-of-sample cumulative growth: fixed vs. two learned distances')
ax.set_ylabel('Cumulative growth of $1')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare SML with correlation separately in each calendar regime.

rows_sml = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    base_c = [base_chunks[i] for i in idx]
    sml_c = [sml_chunks[i] for i in idx]
    base_r = pd.concat(base_c)
    sml_r = pd.concat(sml_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(sml_c, base_c)
    rows_sml.append({
        'Regime': name,
        'N windows': len(idx),
        'Corr-HRP Sharpe': sharpe_lo2002(base_r),
        'SML-HRP Sharpe': sharpe_lo2002(sml_r),
        'Delta Sharpe': obs,
        '95% CI lo': lo_ci,
        '95% CI hi': hi_ci,
        'p-value': p,
    })

regime_table_sml = pd.DataFrame(rows_sml).set_index('Regime')
print("=== Regime sub-split: SML-HRP vs Correlation-HRP ===")
print(regime_table_sml.round(4).to_string())

In [ ]:
# Correct the current regime comparisons as one family using Holm.
# Keep full precision; this does not validate the underlying bootstrap test.
from statsmodels.stats.multitest import multipletests

comparison_tables_a = {'AE': regime_table, 'SML': regime_table_sml}
pval_labels_a, pvals_a = [], []
for method, table in comparison_tables_a.items():
    for regime, value in table['p-value'].items():
        pval_labels_a.append(f'{method} [{regime}]')
        pvals_a.append(value)

pvals_a = np.asarray(pvals_a, dtype=float)
if pvals_a.size == 0 or not np.all(np.isfinite(pvals_a) & (pvals_a >= 0) & (pvals_a <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')
# Apply Holm at family-wise alpha=0.05 to this cell’s collected comparisons.
reject_a, pvals_holm_a, _, _ = multipletests(pvals_a, alpha=0.05, method='holm')
holm_df_a = pd.DataFrame(
    {'p_raw': pvals_a, 'p_holm': pvals_holm_a, 'survives_0.05': reject_a},
    index=pval_labels_a,
)
print(f'=== Holm correction: learned versus correlation, {len(pvals_a)} tests ===')
print(holm_df_a.round(4).to_string())


## 6. Comparisons with PCA

Recompute three-component PCA distances and HRP allocations on the same windows. PCA is the selected Phase 1 comparator. Selection using the same evaluation history should be acknowledged when interpreting the later comparisons.


In [ ]:
# Fit PCA to standardized training returns.
# Use asset entries in the retained component vectors for Euclidean distances.

from sklearn.decomposition import PCA

def get_pca_distance(train_returns,n_components=3):
    X = train_returns.values
    # Standardize each asset’s training returns before fitting PCA.
    # Rows are dates and columns are assets; the outer test data is excluded.
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    pca = PCA(n_components=n_components)
    pca.fit(X_std)
    # Use asset entries in the retained component vectors (N by n_components).
    # These coordinates are not multiplied by explained-variance eigenvalues.
    loadings = pca.components_.T
    n = loadings.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        diff = loadings - loadings[i]
        D[i] = np.sqrt((diff**2).sum(axis=1))
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=train_returns.columns, columns=train_returns.columns)

print("PCA distance (Phase 1 champion) defined.")

In [ ]:
# Recompute PCA-HRP on the same windows for aligned paired comparisons.

pca_rets, pca_chunks, pca_turnover = [], [], []
prev_w_pca = None
for ts, te, vs, ve in windows:
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    dist = get_pca_distance(tr)
    w = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ w[SYMBOLS]).rename('PCA-HRP')
    pca_rets.append(r); pca_chunks.append(r)
    if prev_w_pca is not None:
        # Record the full absolute change between consecutive target allocations.
        pca_turnover.append(np.abs(w - prev_w_pca).sum())
    prev_w_pca = w

pca_series = pd.concat(pca_rets)
print("=== PCA-HRP (Phase 1 champion, re-run here for chunk alignment) ===")
print(f"Sharpe (Lo 2002): {sharpe_lo2002(pca_series):.3f} (sanity check vs Phase 1's 1.0511)")
print(f"Max Drawdown: {max_drawdown(pca_series):.4f}")
print(f"CEQ (gamma=1): {ceq(pca_series, 1):.4f}")
print(f"Avg quarterly turnover: {np.mean(pca_turnover):.4f}")

In [ ]:
# Positive deltas favour the learned method, since each call is candidate minus PCA.
# A confidence interval spanning zero does not establish equivalence.

# Compare AE and SML with PCA using mean paired window Sharpe differences.

obs_ae, lo_ae, hi_ae, p_ae = bootstrap_sharpe_diff(ae_chunks, pca_chunks)
obs_sml, lo_sml, hi_sml, p_sml = bootstrap_sharpe_diff(sml_chunks, pca_chunks)

print("=== AE-Embedding-HRP vs PCA-HRP (Phase 1 champion) ===")
print(f"Delta Sharpe: {obs_ae:+.4f}  95% CI [{lo_ae:+.4f}, {hi_ae:+.4f}]  p-value: {p_ae:.4f}")

print("=== SML-HRP vs PCA-HRP (Phase 1 champion) ===")
print(f"Delta Sharpe: {obs_sml:+.4f}  95% CI [{lo_sml:+.4f}, {hi_sml:+.4f}]  p-value: {p_sml:.4f}")

In [ ]:
# Each row uses pooled daily returns; average turnover uses successive target
# changes and therefore normally contains one fewer observation than the window count.

# Display pooled performance across the four main methods.

summary_pca = pd.DataFrame({
    'Sharpe': [sharpe_lo2002(pca_series), sharpe_lo2002(base_series), sharpe_lo2002(ae_series), sharpe_lo2002(sml_series)],
    'MaxDD': [max_drawdown(pca_series), max_drawdown(base_series), max_drawdown(ae_series), max_drawdown(sml_series)],
    'CEQ (g=1)': [ceq(pca_series, 1), ceq(base_series, 1), ceq(ae_series, 1), ceq(sml_series, 1)],
    'Avg Turnover': [np.mean(pca_turnover), np.mean(base_turnover), np.mean(ae_turnover), np.mean(sml_turnover)],
}, index=['PCA-HRP (Phase 1 champion)', 'Correlation-HRP', 'AE-Embedding-HRP', 'SML-HRP'])

print("=== Summary vs PCA-HRP (Phase 1 champion) baseline, 79 windows ===")
print(summary_pca.round(4).to_string())

In [ ]:
# Report pooled regime metrics and paired window Sharpe differences versus PCA.

rows_ae_pca = []
rows_sml_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    ae_c = [ae_chunks[i] for i in idx]
    sml_c = [sml_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    ae_r = pd.concat(ae_c)
    sml_r = pd.concat(sml_c)
    obs_a, lo_a, hi_a, p_a = bootstrap_sharpe_diff(ae_c, pca_c)
    obs_s, lo_s, hi_s, p_s = bootstrap_sharpe_diff(sml_c, pca_c)
    row_a = {'Regime': name, 'N windows': len(idx)}
    row_a['PCA-HRP Sharpe'] = sharpe_lo2002(pca_r)
    row_a['AE-HRP Sharpe'] = sharpe_lo2002(ae_r)
    row_a['Delta Sharpe'] = obs_a
    row_a['95% CI lo'] = lo_a
    row_a['95% CI hi'] = hi_a
    row_a['p-value'] = p_a
    rows_ae_pca.append(row_a)
    row_s = {'Regime': name, 'N windows': len(idx)}
    row_s['PCA-HRP Sharpe'] = sharpe_lo2002(pca_r)
    row_s['SML-HRP Sharpe'] = sharpe_lo2002(sml_r)
    row_s['Delta Sharpe'] = obs_s
    row_s['95% CI lo'] = lo_s
    row_s['95% CI hi'] = hi_s
    row_s['p-value'] = p_s
    rows_sml_pca.append(row_s)

regime_table_ae_pca = pd.DataFrame(rows_ae_pca).set_index('Regime')
regime_table_sml_pca = pd.DataFrame(rows_sml_pca).set_index('Regime')

print("=== Regime sub-split: AE-Embedding-HRP vs PCA-HRP ===")
print(regime_table_ae_pca.round(4).to_string())

print("=== Regime sub-split: SML-HRP vs PCA-HRP ===")
print(regime_table_sml_pca.round(4).to_string())

In [ ]:
# Correct the current regime comparisons as one family using Holm.
# Keep full precision; this does not validate the underlying bootstrap test.
from statsmodels.stats.multitest import multipletests

comparison_tables_b = {'AE': regime_table_ae_pca, 'SML': regime_table_sml_pca}
pval_labels_b, pvals_b = [], []
for method, table in comparison_tables_b.items():
    for regime, value in table['p-value'].items():
        pval_labels_b.append(f'{method} [{regime}]')
        pvals_b.append(value)

pvals_b = np.asarray(pvals_b, dtype=float)
if pvals_b.size == 0 or not np.all(np.isfinite(pvals_b) & (pvals_b >= 0) & (pvals_b <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')
# Apply Holm at family-wise alpha=0.05 to this cell’s collected comparisons.
reject_b, pvals_holm_b, _, _ = multipletests(pvals_b, alpha=0.05, method='holm')
holm_df_b = pd.DataFrame(
    {'p_raw': pvals_b, 'p_holm': pvals_holm_b, 'survives_0.05': reject_b},
    index=pval_labels_b,
)
print(f'=== Holm correction: learned versus PCA, {len(pvals_b)} tests ===')
print(holm_df_b.round(4).to_string())


## 7. Shrinkage toward PCA allocations

* Add a squared-distance penalty between the resulting portfolio weights and training-window PCA-HRP weights.
* The penalty acts on portfolio allocations, not directly on feature weights.
* Compare the lambda grid as an exploratory sensitivity study; selecting lambda from these test results is not independent validation.


In [ ]:
# Penalize squared differences between HRP portfolio allocations and PCA allocations.
# The penalty acts on holdings, not directly on feature-distance weights.

def insample_variance_shrink_objective(theta, F, train_returns, w_pca, lam):
    w = theta**2
    if w.sum() < 1e-8:
        return 1e6
    dist = weighted_feature_distance(F, w)
    try:
        wgt = hrp_weights_from_dist(train_returns, dist)
    except Exception:
        return 1e6
    cov = train_returns.cov()
    var = wgt.values @ cov.values @ wgt.values
    # Add lambda times squared Euclidean distance between portfolio allocations.
    # The variance term is daily variance, so lambda’s effect depends on that scale.
    # Both holdings arrays must already have identical asset order.
    penalty = lam * np.sum((wgt.values - w_pca.values) ** 2)
    return float(var + penalty)

def fit_metric_weights_shrink(train_returns, w_pca, lam, n_restarts=4, maxiter=60, seed=0):
    feats_df = build_asset_features(train_returns)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=len(FEATURE_NAMES))
        res = minimize(insample_variance_shrink_objective, theta0, args=(feats_df, train_returns, w_pca, lam),
                        method='Nelder-Mead',
                        options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    if best is None:
        return feats_df, np.ones(len(FEATURE_NAMES))
    return feats_df, best[1]

print("SML-Shrink functions defined.")

In [ ]:
# Cache training-window PCA allocations for shrinkage fits and diagnostics.

# Cache one ticker-indexed weight vector per window, in windows order.
# Later fits retrieve the matching reference by k instead of using test returns.
pca_weights_per_window = []
for (ts, te, vs, ve) in windows:
    tr = returns.loc[ts:te]
    pca_dist = get_pca_distance(tr)
    w_pca = hrp_weights_from_dist(tr, pca_dist)
    pca_weights_per_window.append(w_pca)
print(f"Cached PCA-HRP weights for {len(pca_weights_per_window)} windows.")

In [ ]:
# Evaluate every lambda on the same windows; selection from these results is exploratory.

# Evaluate all five settings on the same outer test history.
# Comparing the resulting scores is a sensitivity study; picking a winner uses those test outcomes.
lam_grid = [0, 0.001, 0.005, 0.01, 0.05]
shrink_results = {}

for lam in lam_grid:
    rets, chunks, turns = [], [], []
    prev_w = None
    for k, (ts, te, vs, ve) in enumerate(windows):
        # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
        # A new target allocation is estimated for each outer training window.
        tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
        w_pca = pca_weights_per_window[k]
        feats_df, w = fit_metric_weights_shrink(tr, w_pca, lam, n_restarts=4, maxiter=60, seed=k)
        dist = weighted_feature_distance(feats_df, w)
        wgt = hrp_weights_from_dist(tr, dist)
        # Apply fixed target weights to each daily return vector in this test window.
        # This is gross constant-weight performance; trading costs are not deducted here.
        r = (tst @ wgt[SYMBOLS]).rename(f'SML-Shrink-HRP(lam={lam})')
        rets.append(r); chunks.append(r)
        if prev_w is not None:
            # Record the full L1 change between consecutive target allocations.
            turns.append(np.abs(wgt - prev_w).sum())
        prev_w = wgt
    series = pd.concat(rets)
    shrink_results[lam] = {'series': series, 'chunks': chunks, 'turnover': turns}
    print(f"lam={lam:<6}  Sharpe={sharpe_lo2002(series):.4f}  MaxDD={max_drawdown(series):.4f}  CEQ(g=1)={ceq(series,1):.4f}  AvgTurnover={np.mean(turns):.4f}")

print("SML-Shrink lambda grid complete.")

In [ ]:
# The helper resamples circular blocks of consecutive paired
# window Sharpe differences using the shared block-length setting.
# The comparisons for different lambda values are not corrected together here.

# Compare each shrinkage setting with both benchmarks. These are raw comparisons.

print("=== SML-Shrink-HRP significance vs Correlation-HRP and PCA-HRP (block bootstrap) ===")
sig_rows = []
for lam in lam_grid:
    chunks_lam = shrink_results[lam]['chunks']
    obs_c, lo_c, hi_c, p_c = bootstrap_sharpe_diff(chunks_lam, base_chunks)
    obs_p, lo_p, hi_p, p_p = bootstrap_sharpe_diff(chunks_lam, pca_chunks)
    sig_rows.append((lam, obs_c, lo_c, hi_c, p_c, obs_p, lo_p, hi_p, p_p))
    print(f"lam={lam} vs Corr dSharpe={obs_c:.4f} CI_lo={lo_c:.4f} CI_hi={hi_c:.4f} p={p_c:.4f} | vs PCA dSharpe={obs_p:.4f} CI_lo={lo_p:.4f} CI_hi={hi_p:.4f} p={p_p:.4f}")

In [ ]:
# Use the manually selected lambda=0.005 for regime comparisons.
# The variable best_lam does not implement an independent selection rule.

# This constant is chosen manually. It does not estimate lambda on a separate
# validation set or prove that 0.005 is optimal.
best_lam = 0.005
shrink_chunks_best = shrink_results[best_lam]['chunks']
rows_shrink_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    shrink_c = [shrink_chunks_best[i] for i in idx]
    pca_r = pd.concat(pca_c)
    shrink_r = pd.concat(shrink_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(shrink_c, pca_c)
    rows_shrink_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-Shrink-HRP Sharpe': sharpe_lo2002(shrink_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p})

regime_table_shrink_pca = pd.DataFrame(rows_shrink_pca).set_index('Regime')
print(f"=== Regime sub-split: SML-Shrink-HRP (lam={best_lam}) vs PCA-HRP ===")
print(regime_table_shrink_pca.round(4).to_string())

## 8. SML with an ERC objective

* Keep the six features, weighted distance and HRP allocator.
* Fit feature weights to reduce squared deviations of relative asset risk contributions from 1/N.
* This encourages equal contributions within the portfolios HRP can produce; it does not guarantee an exact ERC allocation.


In [ ]:
# Encourage relative risk contributions close to 1/N through learned distances.
# HRP remains the allocator; exact equal risk contribution is not guaranteed.

def erc_objective(theta, F, train_returns):
    w = theta**2
    if w.sum() < 1e-8:
        return 1e6
    dist = weighted_feature_distance(F, w)
    try:
        wgt = hrp_weights_from_dist(train_returns, dist)
    except Exception:
        return 1e6
    cov = train_returns.cov()
    wv = wgt.values
    port_var = wv @ cov.values @ wv
    if port_var < 1e-12:
        return 1e6
    # cov @ holdings gives marginal contributions to the variance quadratic form.
    marginal = cov.values @ wv
    # Normalize each w_i*(cov @ w)_i by total variance. Contributions sum to one
    # when variance is nonzero, but an individual contribution can be negative.
    rc = wv * marginal / port_var
    n = len(wv)
    # Penalize deviation from equal relative risk contribution 1/N.
    # Only distances are optimized; the feasible allocations remain those produced by HRP.
    return float(np.sum((rc - 1.0/n)**2))

def fit_metric_weights_erc(train_returns, n_restarts=4, maxiter=60, seed=0):
    feats_df = build_asset_features(train_returns)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=len(FEATURE_NAMES))
        res = minimize(erc_objective, theta0, args=(feats_df, train_returns), method='Nelder-Mead', options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    if best is None:
        return feats_df, np.ones(len(FEATURE_NAMES))
    return feats_df, best[1]

print('SML-ERC functions defined.')

In [ ]:
# Fit the ERC distance objective on training data and evaluate the next window.

erc_rets, erc_chunks, erc_turnover, erc_weights_log = [], [], [], []
prev_w_erc = None
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    feats_df, w = fit_metric_weights_erc(tr, n_restarts=4, maxiter=60, seed=k)
    erc_weights_log.append(w)
    dist = weighted_feature_distance(feats_df, w)
    wgt = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ wgt[SYMBOLS]).rename('SML-ERC-HRP')
    erc_rets.append(r); erc_chunks.append(r)
    if prev_w_erc is not None:
        erc_turnover.append(np.abs(wgt - prev_w_erc).sum())
    prev_w_erc = wgt
erc_series = pd.concat(erc_rets)
print(f'SML-ERC-HRP: Sharpe={sharpe_lo2002(erc_series):.4f} MaxDD={max_drawdown(erc_series):.4f} CEQ(g=1)={ceq(erc_series,1):.4f} AvgTurnover={np.mean(erc_turnover):.4f}')
print(f'erc_weights_log length: {len(erc_weights_log)}')

In [ ]:
# Both calls evaluate the same ERC candidate against different comparators.
# These are raw separate comparisons, not a corrected two-test family.

# Compare SML-ERC with correlation and PCA using the same paired-window procedure.

obs_erc_corr, lo_erc_corr, hi_erc_corr, p_erc_corr = bootstrap_sharpe_diff(erc_chunks, base_chunks)
obs_erc_pca, lo_erc_pca, hi_erc_pca, p_erc_pca = bootstrap_sharpe_diff(erc_chunks, pca_chunks)
print(f'SML-ERC-HRP vs Correlation-HRP: dSharpe={obs_erc_corr:+.4f} CI=[{lo_erc_corr:+.4f},{hi_erc_corr:+.4f}] p={p_erc_corr:.4f}')
print(f'SML-ERC-HRP vs PCA-HRP: dSharpe={obs_erc_pca:+.4f} CI=[{lo_erc_pca:+.4f},{hi_erc_pca:+.4f}] p={p_erc_pca:.4f}')

In [ ]:
# Compare SML-ERC with PCA inside GFC and COVID+ calendar windows.

gfc_idx = [i for i, r in enumerate(regime_labels) if r == 'GFC']
covid_idx = [i for i, r in enumerate(regime_labels) if r == 'COVID+']
gfc_erc = [erc_chunks[i] for i in gfc_idx]
gfc_pca = [pca_chunks[i] for i in gfc_idx]
covid_erc = [erc_chunks[i] for i in covid_idx]
covid_pca = [pca_chunks[i] for i in covid_idx]
obs_g, lo_g, hi_g, p_g = bootstrap_sharpe_diff(gfc_erc, gfc_pca)
obs_c, lo_c, hi_c, p_c = bootstrap_sharpe_diff(covid_erc, covid_pca)
print(f"GFC: dSharpe={obs_g:+.4f} CI=[{lo_g:+.4f},{hi_g:+.4f}] p={p_g:.4f}")
print(f"COVID+: dSharpe={obs_c:+.4f} CI=[{lo_c:+.4f},{hi_c:+.4f}] p={p_c:.4f}")

### 8.1 Direct ERC-versus-variance comparison

Compare SML-ERC-HRP directly with SML-HRP overall, in GFC and in COVID+. The bootstrap operates on paired window Sharpe differences.


In [ ]:
gfc_idx = [i for i, r in enumerate(regime_labels) if r == 'GFC']
covid_idx = [i for i, r in enumerate(regime_labels) if r == 'COVID+']
overall = bootstrap_sharpe_diff(erc_chunks, sml_chunks, n_boot=5000, seed=42)
gfc_erc = [erc_chunks[i] for i in gfc_idx]
gfc_sml = [sml_chunks[i] for i in gfc_idx]
gfc = bootstrap_sharpe_diff(gfc_erc, gfc_sml, n_boot=5000, seed=42)
covid_erc = [erc_chunks[i] for i in covid_idx]
covid_sml = [sml_chunks[i] for i in covid_idx]
covid = bootstrap_sharpe_diff(covid_erc, covid_sml, n_boot=5000, seed=42)
print('=== SML-ERC-HRP vs SML-HRP direct paired circular block bootstrap ===')
print('Overall (n=79): mean_diff, ci_lo, ci_hi, p =', overall)
print('GFC (n=8):      mean_diff, ci_lo, ci_hi, p =', gfc)
print('COVID+ (n=25):  mean_diff, ci_lo, ci_hi, p =', covid)

## 9. CMA-ES optimizer comparison

Retain the variance objective and replace Nelder-Mead with CMA-ES. Initialization and search budgets differ, so compare objective values and evaluations before calling either optimizer better. Check seed semantics in the installed CMA-ES version, particularly for seed=0.


In [ ]:
# Use CMA-ES on the unchanged in-sample variance objective.
# Its evaluation budget does not guarantee convergence or a global optimum.

!pip install cma --quiet
import cma

def fit_metric_weights_cmaes(train_returns, sigma0=0.5, maxfevals=300, seed=0):
    feats_df = build_asset_features(train_returns)
    # Initialize all feature parameters at one. sigma0 sets the initial search spread;
    # maxfevals limits objective evaluations, not the number of outer test windows.
    theta0 = np.ones(len(FEATURE_NAMES))
    es = cma.CMAEvolutionStrategy(theta0, sigma0, {'maxfevals': maxfevals, 'seed': seed, 'verbose': -9})
    es.optimize(lambda th: insample_variance_objective(np.array(th), feats_df, train_returns))
    # Take the optimizer’s best evaluated parameter vector and square it to obtain
    # nonnegative distance weights. Termination status is not inspected here.
    theta_best = es.result.xbest
    w = np.asarray(theta_best) ** 2
    if w.sum() < 1e-8:
        w = np.ones(len(FEATURE_NAMES))
    return feats_df, w

print("CMA-ES SML function defined.")

In [ ]:
# Fit CMA-ES distances using training returns and evaluate subsequent test windows.

cmaes_rets, cmaes_chunks, cmaes_turnover = [], [], []
prev_w_cmaes = None
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    feats_df, w = fit_metric_weights_cmaes(tr, sigma0=0.5, maxfevals=300, seed=k)
    dist = weighted_feature_distance(feats_df, w)
    wgt = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ wgt[SYMBOLS]).rename('SML-CMAES-HRP')
    cmaes_rets.append(r); cmaes_chunks.append(r)
    if prev_w_cmaes is not None:
        # This excludes initial entry, weight drift and trading to maintain targets within a window.
        cmaes_turnover.append(np.abs(wgt - prev_w_cmaes).sum())
    prev_w_cmaes = wgt
cmaes_series = pd.concat(cmaes_rets)
print(f"SML-CMAES-HRP: Sharpe={sharpe_lo2002(cmaes_series):.4f} MaxDD={max_drawdown(cmaes_series):.4f} CEQ(g=1)={ceq(cmaes_series,1):.4f} AvgTurnover={np.mean(cmaes_turnover):.4f}")

In [ ]:
# Compare CMA-ES with correlation and PCA using paired window differences.

obs_cmaes_corr, lo_cmaes_corr, hi_cmaes_corr, p_cmaes_corr = bootstrap_sharpe_diff(cmaes_chunks, base_chunks)
obs_cmaes_pca, lo_cmaes_pca, hi_cmaes_pca, p_cmaes_pca = bootstrap_sharpe_diff(cmaes_chunks, pca_chunks)
print(f'SML-CMAES-HRP vs Correlation-HRP: dSharpe={obs_cmaes_corr:+.4f} CI=[{lo_cmaes_corr:+.4f},{hi_cmaes_corr:+.4f}] p={p_cmaes_corr:.4f}')
print(f'SML-CMAES-HRP vs PCA-HRP: dSharpe={obs_cmaes_pca:+.4f} CI=[{lo_cmaes_pca:+.4f},{hi_cmaes_pca:+.4f}] p={p_cmaes_pca:.4f}')

In [ ]:
# Repeat the CMA-ES versus PCA comparison within market regimes.

rows_cmaes_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    cmaes_c = [cmaes_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    cmaes_r = pd.concat(cmaes_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(cmaes_c, pca_c)
    rows_cmaes_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-CMAES-HRP Sharpe': sharpe_lo2002(cmaes_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p})
regime_table_cmaes_pca = pd.DataFrame(rows_cmaes_pca).set_index('Regime')
print('=== Regime sub-split: SML-CMAES-HRP vs PCA-HRP ===')
print(regime_table_cmaes_pca.round(4).to_string())

### 9.1 Bootstrap seed sensitivity

Repeat the comparison with one additional resampling seed. This checks bootstrap randomness, not the randomness of fitting CMA-ES. One additional seed does not establish general robustness.


In [ ]:
# Compare resampling seeds using current fitted returns.
for seed in (42, 999):
    result = bootstrap_sharpe_diff(cmaes_chunks, pca_chunks, seed=seed)
    print(f"CMA-ES versus PCA, seed={seed}: delta, CI low, CI high, p = {result}")


## 10. Alternative cluster-risk allocator

* The functions named HERC below implement a custom HRP variant: they retain midpoint bisection and replace inverse-variance cluster risk with pseudoinverse-based minimum-variance cluster risk.
* SML distances are still fitted using the original HRP variance objective, then transferred to the modified allocator.

The historical HERC function names refer to this custom allocator, not the published HERC algorithm. Its results are an allocator-transfer check, not a full refit under that allocator.


In [ ]:
# Estimate cluster risk with unconstrained minimum-variance weights.
# Internal minimum-variance cluster weights may be negative.

def cluster_var_minvar(cov_slice):
    # Use the pseudoinverse to handle singular covariance matrices. The resulting
    # cluster portfolio is unconstrained and can contain negative component weights.
    inv_cov = np.linalg.pinv(cov_slice.values)
    ones = np.ones(len(cov_slice))
    w = inv_cov @ ones
    # Normalize the minimum-variance direction to sum to one. A zero or unstable
    # denominator is not guarded against in this implementation.
    w = w / w.sum()
    return w @ cov_slice.values @ w

def get_rec_bipart_herc(cov, sort_ix):
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix]
    while len(c_items) > 0:
        c_items = [i[j:k] for i in c_items for j, k in ((0, len(i)//2), (len(i)//2, len(i))) if len(i) > 1]
        for i in range(0, len(c_items), 2):
            c0, c1 = c_items[i], c_items[i+1]
            w0 = cluster_var_minvar(cov.loc[c0, c0])
            w1 = cluster_var_minvar(cov.loc[c1, c1])
            # Use minimum-variance cluster risk in the same inverse-risk split as HRP.
            # Internal cluster weights estimate risk; they are not directly assigned as final holdings.
            alpha = 1 - w0 / (w0 + w1)
            w[c0] *= alpha
            w[c1] *= 1 - alpha
    return w

def herc_weights_from_dist(returns_df, dist_df):
    cov = returns_df.cov()
    condensed = squareform(dist_df.values, checks=False)
    link = linkage(condensed, method='single')
    sort_ix_pos = quasi_diag(link, len(returns_df.columns))
    sort_ix = [returns_df.columns[i] for i in sort_ix_pos]
    w = get_rec_bipart_herc(cov, sort_ix)
    return (w / w.sum()).reindex(SYMBOLS)

print("HERC allocator (risk-parity bisection using full min-variance cluster risk) defined.")

In [ ]:
# Evaluate correlation and PCA with the same modified cluster-risk allocator.

corr_herc_rets, corr_herc_chunks = [], []
pca_herc_rets, pca_herc_chunks = [], []
for (ts, te, vs, ve) in windows:
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    d1 = get_corr_distance(tr)
    w1 = herc_weights_from_dist(tr, d1)
    r1 = (tst @ w1[SYMBOLS]).rename('Correlation-HERC')
    corr_herc_rets.append(r1); corr_herc_chunks.append(r1)
    d2 = get_pca_distance(tr)
    w2 = herc_weights_from_dist(tr, d2)
    r2 = (tst @ w2[SYMBOLS]).rename('PCA-HERC')
    pca_herc_rets.append(r2); pca_herc_chunks.append(r2)
corr_herc_series = pd.concat(corr_herc_rets)
pca_herc_series = pd.concat(pca_herc_rets)
print(f"Correlation-HERC: Sharpe={sharpe_lo2002(corr_herc_series):.4f} MaxDD={max_drawdown(corr_herc_series):.4f}")
print(f"PCA-HERC: Sharpe={sharpe_lo2002(pca_herc_series):.4f} MaxDD={max_drawdown(pca_herc_series):.4f}")

In [ ]:
# Fit SML distances against the original HRP objective, then transfer to the variant.

sml_herc_rets, sml_herc_chunks = [], []
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    feats_df, w = fit_metric_weights(tr, n_restarts=4, maxiter=60, seed=k)
    dist = weighted_feature_distance(feats_df, w)
    wgt = herc_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ wgt[SYMBOLS]).rename('SML-HERC')
    sml_herc_rets.append(r); sml_herc_chunks.append(r)
sml_herc_series = pd.concat(sml_herc_rets)
print(f"SML-HERC: Sharpe={sharpe_lo2002(sml_herc_series):.4f} MaxDD={max_drawdown(sml_herc_series):.4f}")
obs_herc, lo_herc, hi_herc, p_herc = bootstrap_sharpe_diff(sml_herc_chunks, pca_herc_chunks)
print(f'SML-HERC vs PCA-HERC: dSharpe={obs_herc:+.4f} CI=[{lo_herc:+.4f},{hi_herc:+.4f}] p={p_herc:.4f}')

In [ ]:
# Compare SML and PCA under the modified allocator by calendar regime.

rows_herc = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_herc_chunks[i] for i in idx]
    sml_c = [sml_herc_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    sml_r = pd.concat(sml_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(sml_c, pca_c)
    rows_herc.append({'Regime': name, 'N windows': len(idx), 'PCA-HERC Sharpe': sharpe_lo2002(pca_r), 'SML-HERC Sharpe': sharpe_lo2002(sml_r), 'Delta Sharpe': obs, '95% CI lo': lo_ci, '95% CI hi': hi_ci, 'p-value': p})
regime_table_herc = pd.DataFrame(rows_herc).set_index('Regime')
print("=== Regime sub-split: SML-HERC vs PCA-HERC (all four regimes) ===")
print(regime_table_herc.round(4).to_string())

In [ ]:
# Holm-bonferroni correction

from statsmodels.stats.multitest import multipletests

comparison_tables_c = {'SML variant versus PCA variant': regime_table_herc}
pval_labels_c, pvals_c = [], []
for method, table in comparison_tables_c.items():
    for regime, value in table['p-value'].items():
        pval_labels_c.append(f'{method} [{regime}]')
        pvals_c.append(value)

pvals_c = np.asarray(pvals_c, dtype=float)
if pvals_c.size == 0 or not np.all(np.isfinite(pvals_c) & (pvals_c >= 0) & (pvals_c <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')

reject_c, pvals_holm_c, _, _ = multipletests(pvals_c, alpha=0.05, method='holm')
holm_df_c = pd.DataFrame(
    {'p_raw': pvals_c, 'p_holm': pvals_holm_c, 'survives_0.05': reject_c},
    index=pval_labels_c,
)
print(f'=== Holm correction: modified allocator, {len(pvals_c)} tests ===')
print(holm_df_c.round(4).to_string())


## 11. ERC weighting of autoencoder embeddings

* First fit the reconstruction autoencoder.
* Then hold embeddings fixed and learn nonnegative weights on their eight coordinates to reduce unequal risk contributions in HRP.
* The neural network is not retrained with an ERC loss.
* Compare AE-ERC directly with AE before claiming an improvement over AE.


In [ ]:
# Fit reconstruction embeddings first, then learn ERC-based coordinate weights.
# The neural network itself is not fitted with an ERC loss.

def get_ae_embedding_matrix(train_returns, L=20, bottleneck_dim=8, seed=0):
    X_ae = build_ae_training_matrix(train_returns, L=L)
    model = fit_autoencoder(X_ae, bottleneck_dim=bottleneck_dim, seed=seed)
    embs = {}
    for col in train_returns.columns:
        s = train_returns[col].values
        mu, sd = s.mean(), s.std() + 1e-12
        embs[col] = encode(model, ((s[-L:] - mu) / sd).reshape(1, -1))[0]
    return pd.DataFrame(embs).T

def fit_ae_erc_weights(train_returns, n_restarts=4, maxiter=60, seed=0):
    # Fit the autoencoder once for this window, then hold the embedding matrix
    # fixed while optimizing coordinate weights across all restarts.
    E = get_ae_embedding_matrix(train_returns, seed=seed)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=E.shape[1])
        # Reuse erc_objective with eight latent coordinates in place of six named
        # features. Gradients are not propagated into the neural network.
        res = minimize(erc_objective, theta0, args=(E, train_returns), method='Nelder-Mead', options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    return E, (best[1] if best is not None else np.ones(E.shape[1]))

print('AE-ERC functions defined.')

In [ ]:
# Evaluate the two-stage AE-plus-ERC-distance model on following test windows.

ae_erc_rets, ae_erc_chunks, ae_erc_turnover = [], [], []
prev_w_ae_erc = None
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    E, w = fit_ae_erc_weights(tr, n_restarts=4, maxiter=60, seed=k)
    dist = weighted_feature_distance(E, w)
    wgt = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    r = (tst @ wgt[SYMBOLS]).rename('AE-ERC-HRP')
    ae_erc_rets.append(r); ae_erc_chunks.append(r)
    if prev_w_ae_erc is not None:
        # Record the full L1 change between consecutive target allocations.
        ae_erc_turnover.append(np.abs(wgt - prev_w_ae_erc).sum())
    prev_w_ae_erc = wgt

ae_erc_series = pd.concat(ae_erc_rets)
print(f'AE-ERC-HRP: Sharpe={sharpe_lo2002(ae_erc_series):.4f} MaxDD={max_drawdown(ae_erc_series):.4f} CEQ(g=1)={ceq(ae_erc_series,1):.4f} AvgTurnover={np.mean(ae_erc_turnover):.4f}')

### 11.1 AE-ERC versus PCA: overall and GFC

Report mean window Sharpes and paired comparisons for all windows and the GFC subset.


In [ ]:
# m() averages one Sharpe per chunk with equal window weight.
# Compare AE-ERC with PCA overall and in GFC

ae_erc_gfc = [ae_erc_chunks[i] for i in gfc_idx]
pca_gfc = [pca_chunks[i] for i in gfc_idx]
print('MEAN_AE_ERC_FULL', m(ae_erc_chunks))
print('MEAN_PCA_FULL', m(pca_chunks))
print('MEAN_AE_ERC_GFC', m(ae_erc_gfc))
print('MEAN_PCA_GFC', m(pca_gfc))
print('check overall diff', m(ae_erc_chunks)-m(pca_chunks))
print('check gfc diff', m(ae_erc_gfc)-m(pca_gfc))
obs,lo,hi,p = bootstrap_sharpe_diff(ae_erc_chunks, pca_chunks)
print('bootstrap overall', obs, lo, hi, p)
obs2,lo2,hi2,p2 = bootstrap_sharpe_diff(ae_erc_gfc, pca_gfc)
print('bootstrap gfc', obs2, lo2, hi2, p2)


## 12. Robustness checks
### 12.1 Chronological split and daily transaction costs
Compare pooled half-sample Sharpes. Simulate daily target rebalancing with proportional costs, including initial entry, for SML-ERC and PCA. This matches the constant-mix gross return convention; it is not quarterly buy-and-hold.


In [ ]:
# Compare chronological halves using pooled return metrics.
# Costs below are applied to daily self-financing trades.

# With 79 windows this gives 39 in the first half and 40 in the second.
# Concatenation preserves full daily series rather than averaging window Sharpes.
n_first = len(windows) // 2
first_erc = pd.concat(erc_chunks[:n_first])
second_erc = pd.concat(erc_chunks[n_first:])
first_pca = pd.concat(pca_chunks[:n_first])
second_pca = pd.concat(pca_chunks[n_first:])
print('=== Chronological Half-Split ===')
print(f'First half ({windows[0][2].date()} to {windows[n_first-1][3].date()}), n={n_first} windows:')
print(f'  SML-ERC-HRP Sharpe: {sharpe_lo2002(first_erc):.4f}')
print(f'  PCA-HRP     Sharpe: {sharpe_lo2002(first_pca):.4f}')
print(f'Second half ({windows[n_first][2].date()} to {windows[-1][3].date()}), n={len(windows)-n_first} windows:')
print(f'  SML-ERC-HRP Sharpe: {sharpe_lo2002(second_erc):.4f}')
print(f'  PCA-HRP     Sharpe: {sharpe_lo2002(second_pca):.4f}')
# Reconstruct SML-ERC allocations from fitted features; no optimiser is rerun.
# Model daily constant-mix rebalancing, consistent with the gross backtest.
# Cost rate applies per dollar bought or sold, includes initial entry from cash,
# excludes final liquidation, spreads beyond the assumed rate, taxes and market impact.
# Solve transaction costs against post-cost target holdings (self-financing).
erc_targets = []
for k, (ts, te, vs, ve) in enumerate(windows):
    train = returns.loc[ts:te]
    erc_targets.append(hrp_weights_from_dist(train, weighted_feature_distance(build_asset_features(train), erc_weights_log[k])))

def simulate_daily_costs(targets, rate):
    wealth = 1.0
    holdings = np.zeros(len(SYMBOLS))
    values, dates = [], []
    for target, (_, _, vs, ve) in zip(targets, windows):
        w = target.reindex(SYMBOLS).to_numpy()
        if not np.isfinite(w).all() or (w < 0).any() or not np.isclose(w.sum(), 1):
            raise ValueError('Invalid long-only target weights.')
        for date, row in returns.loc[vs:ve, SYMBOLS].iterrows():
            before = wealth
            lo, hi = 0.0, before
            for _ in range(60):
                after = (lo + hi) / 2
                balance = after + rate * np.abs(after*w - holdings).sum() - before
                if balance > 0: hi = after
                else: lo = after
            after = (lo + hi) / 2
            holdings = after*w*(1 + row.to_numpy())
            wealth = holdings.sum()
            values.append(wealth/before - 1)
            dates.append(date)
    return pd.Series(values, index=pd.DatetimeIndex(dates, name=returns.index.name))

cost_results = []
for name, targets, gross in [('SML-ERC', erc_targets, erc_series), ('PCA', pca_weights_per_window, pca_series)]:
    zero_cost = simulate_daily_costs(targets, 0.0)
    pd.testing.assert_index_equal(zero_cost.index, gross.index)
    np.testing.assert_allclose(zero_cost.values, gross.values, atol=1e-12, rtol=1e-9)
    for rate in (0.0, 0.0005, 0.0010):
        net = simulate_daily_costs(targets, rate)
        cost_results.append({'Method': name, 'Cost bps per dollar traded': rate*10000,
                             'Sharpe': sharpe_lo2002(net), 'MaxDD': max_drawdown(net),
                             'CEQ': ceq(net, 1), 'Terminal wealth': (1+net).prod()})
print(pd.DataFrame(cost_results).to_string(index=False))


### 12.2 Exploratory volatility-dependent shrinkage

* Rank current training volatility against an expanding history including the current window.
* Rank at least 0.75 selects lambda=0.05; otherwise use 0.005.


In [ ]:
# Rank current training volatility within an expanding history including itself.
# Choose high lambda at rank >= 0.75; the first window always qualifies.
# This volatility-dependent shrinkage is exploratory.

LAM_HIGH, LAM_LOW = 0.05, 0.005
rshrink_rets, rshrink_chunks, rshrink_turns, rshrink_lams, vols_so_far = [], [], [], [], []
prev_w_rshrink = None
for k, (ts, te, vs, ve) in enumerate(windows):
    # Slice inclusive boundaries for this fold: fit on tr, evaluate only on tst.
    # A new target allocation is estimated for each outer training window.
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    # Average the individual assets’ training volatilities, then annualize.
    # This is not the volatility of an equal-weight portfolio.
    cur_vol = tr.std().mean() * np.sqrt(252); vols_so_far.append(cur_vol)
    # The current observation is already in vols_so_far. Its rank uses <=,
    # so ties count as below-or-equal and the first observation has percentile 1.
    pct = (np.array(vols_so_far) <= cur_vol).mean(); lam_k = LAM_HIGH if pct >= 0.75 else LAM_LOW; rshrink_lams.append(lam_k)
    w_pca = pca_weights_per_window[k]
    feats_df, w = fit_metric_weights_shrink(tr, w_pca, lam_k, n_restarts=4, maxiter=60, seed=k)
    dist = weighted_feature_distance(feats_df, w); wgt = hrp_weights_from_dist(tr, dist)
    # Apply fixed target weights to each daily return vector in this test window.
    # This is gross constant-weight performance; trading costs are not deducted here.
    r = (tst @ wgt[SYMBOLS]).rename('SML-Regime-Shrink-HRP'); rshrink_rets.append(r); rshrink_chunks.append(r)
    # Record the full L1 change between consecutive target allocations.
    # This excludes initial entry, weight drift and trading to maintain targets within a window.
    if prev_w_rshrink is not None: rshrink_turns.append(np.abs(wgt - prev_w_rshrink).sum())
    prev_w_rshrink = wgt
rshrink_series = pd.concat(rshrink_rets)
print(f"SML-Regime-Shrink-HRP: Sharpe={sharpe_lo2002(rshrink_series):.4f} MaxDD={max_drawdown(rshrink_series):.4f} CEQ(g=1)={ceq(rshrink_series,1):.4f} AvgTurnover={np.mean(rshrink_turns):.4f}")
print(f"Lambda buckets: HIGH(0.05)={sum(1 for l in rshrink_lams if l==LAM_HIGH)} windows, LOW(0.005)={sum(1 for l in rshrink_lams if l==LAM_LOW)} windows")
obs_rc, lo_rc, hi_rc, p_rc = bootstrap_sharpe_diff(rshrink_chunks, base_chunks)
obs_rp, lo_rp, hi_rp, p_rp = bootstrap_sharpe_diff(rshrink_chunks, pca_chunks)
print(f"SML-Regime-Shrink-HRP vs Correlation-HRP: dSharpe={obs_rc:+.4f} CI=[{lo_rc:+.4f},{hi_rc:+.4f}] p={p_rc:.4f}")
print(f"SML-Regime-Shrink-HRP vs PCA-HRP: dSharpe={obs_rp:+.4f} CI=[{lo_rp:+.4f},{hi_rp:+.4f}] p={p_rp:.4f}")
rows_rshrink_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0: continue
    pca_c = [pca_chunks[i] for i in idx]; rshrink_c = [rshrink_chunks[i] for i in idx]; lams_in = [rshrink_lams[i] for i in idx]
    pca_r = pd.concat(pca_c); rshrink_r = pd.concat(rshrink_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(rshrink_c, pca_c)
    rows_rshrink_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-Regime-Shrink-HRP Sharpe': sharpe_lo2002(rshrink_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p, 'N_high_lam': sum(1 for l in lams_in if l==LAM_HIGH)})
regime_table_rshrink_pca = pd.DataFrame(rows_rshrink_pca).set_index('Regime')
print("=== Regime sub-split: SML-Regime-Shrink-HRP vs PCA-HRP ===")
print(regime_table_rshrink_pca.round(4).to_string())
print('GFC lam values:', [rshrink_lams[i] for i in gfc_idx])
print('COVID+ lam values:', [rshrink_lams[i] for i in covid_idx])

## 13. Weight and concentration diagnostics

### 13.1 Fitted feature weights

* Reuse the variance and ERC feature weights.
* Raw paths have arbitrary common scale: multiplying all feature weights by the same positive constant rescales distances without changing their ordering.
* Use within-window normalised weights for relative emphasis, uniformity and entropy.


In [ ]:
# Reuse fitted feature weights to describe variability, uniformity and entropy.
# Normalize weights for comparisons because overall feature-weight scale is arbitrary.
# These tests do not account for dependence from overlapping training windows.

import numpy as np
import pandas as pd
from scipy import stats

# Each row stores one fit; columns follow FEATURE_NAMES. These parameters
# control distances and must not be interpreted as asset allocation weights.
W_var = np.array(learned_weights_log)   # shape (79, 6), variance-objective SML-HRP
W_erc = np.array(erc_weights_log)       # shape (79, 6), ERC-objective SML-ERC-HRP
print('W_var shape:', W_var.shape, ' W_erc shape:', W_erc.shape)
assert W_var.shape == W_erc.shape == (len(windows), len(FEATURE_NAMES))

def cv(x):
    m = x.mean()
    return np.nan if abs(m) < 1e-12 else x.std(ddof=1) / m

stability_rows = []
for j, feat in enumerate(FEATURE_NAMES):
    stability_rows.append({
        'feature': feat,
        'var_obj_mean': W_var[:, j].mean(), 'var_obj_std': W_var[:, j].std(ddof=1), 'var_obj_CV': cv(W_var[:, j]),
        'erc_obj_mean': W_erc[:, j].mean(), 'erc_obj_std': W_erc[:, j].std(ddof=1), 'erc_obj_CV': cv(W_erc[:, j]),
    })
stability_df = pd.DataFrame(stability_rows).set_index('feature')
print()
print('=== Per-feature weight stability across 79 windows (raw fitted weights, w = theta**2) ===')
print(stability_df.round(4).to_string())
print()
print('Mean CV (variance-objective):', round(stability_df['var_obj_CV'].mean(), 4))
print('Mean CV (ERC-objective):     ', round(stability_df['erc_obj_CV'].mean(), 4))

n_feat = len(FEATURE_NAMES)
uniform = np.ones(n_feat) / n_feat

def normalize_rows(W):
    s = W.sum(axis=1, keepdims=True)
    # Normalize each window independently so relative feature emphasis sums to one.
    # This helper assumes each row has a positive, finite total.
    return W / s

Wn_var = normalize_rows(W_var)
Wn_erc = normalize_rows(W_erc)

def dist_from_uniform(Wn):
    return np.sqrt(((Wn - uniform) ** 2).sum(axis=1))

def entropy_rows(Wn):
    Wn_safe = np.clip(Wn, 1e-12, None)
    # Compute Shannon entropy in natural-log units (nats). Uniform weights attain
    # log(number of features); lower entropy indicates greater feature concentration.
    return -(Wn_safe * np.log(Wn_safe)).sum(axis=1)

d_var = dist_from_uniform(Wn_var)
d_erc = dist_from_uniform(Wn_erc)
h_var = entropy_rows(Wn_var)
h_erc = entropy_rows(Wn_erc)
h_max = np.log(n_feat)

print()
print('=== Per-window distance-from-uniform (Euclidean, weights normalized to sum to 1) ===')
print(f'Variance-objective SML-HRP: mean={d_var.mean():.4f} median={np.median(d_var):.4f} std={d_var.std(ddof=1):.4f}')
print(f'ERC-objective SML-ERC-HRP:  mean={d_erc.mean():.4f} median={np.median(d_erc):.4f} std={d_erc.std(ddof=1):.4f}')

print()
print(f'=== Per-window entropy of normalized weights (nats; max = ln(6) = {h_max:.4f} at perfect uniform) ===')
print(f'Variance-objective SML-HRP: mean={h_var.mean():.4f} median={np.median(h_var):.4f} std={h_var.std(ddof=1):.4f}  (frac of max: {h_var.mean()/h_max:.4f})')
print(f'ERC-objective SML-ERC-HRP:  mean={h_erc.mean():.4f} median={np.median(h_erc):.4f} std={h_erc.std(ddof=1):.4f}  (frac of max: {h_erc.mean()/h_max:.4f})')

# The Wilcoxon alternatives are one-sided; ttest_rel uses its default two-sided
# alternative. These are unadjusted tests and do not model overlapping-window dependence.
w_dist = stats.wilcoxon(d_var, d_erc, alternative='greater')
t_dist = stats.ttest_rel(d_var, d_erc)
w_ent = stats.wilcoxon(h_erc, h_var, alternative='greater')
t_ent = stats.ttest_rel(h_erc, h_var)

print()
print('=== Paired tests across the 79 windows (variance-objective vs ERC-objective) ===')
print(f'Wilcoxon H1: dist_from_uniform(variance) > dist_from_uniform(ERC): stat={w_dist.statistic:.2f} p={w_dist.pvalue:.4g}')
print(f'Paired t-test distance-from-uniform (var - erc): mean_diff={(d_var-d_erc).mean():.4f} t={t_dist.statistic:.3f} p={t_dist.pvalue:.4g}')
print(f'Wilcoxon H1: entropy(ERC) > entropy(variance): stat={w_ent.statistic:.2f} p={w_ent.pvalue:.4g}')
print(f'Paired t-test entropy (erc - var): mean_diff={(h_erc-h_var).mean():.4f} t={t_ent.statistic:.3f} p={t_ent.pvalue:.4g}')

n_var_closer = int((d_var > d_erc).sum())
print()
print(f'Windows where ERC weights are closer to uniform than variance weights: {n_var_closer} / {len(d_var)} ({100*n_var_closer/len(d_var):.1f}%)')


In [ ]:
# Each panel shows a named feature’s RAW fitted weight through time.
# Because common scale is arbitrary, use the normalized diagnostics above
# when comparing relative emphasis between the two objectives.

# Plot saved raw feature weights. Normalized weights better show relative importance.

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
x = np.arange(W_var.shape[0])
for j, feat in enumerate(FEATURE_NAMES):
    ax = axes[j // 3, j % 3]
    ax.plot(x, W_var[:, j], color='tab:blue', lw=1.2, label='Variance-objective (SML-HRP)')
    ax.plot(x, W_erc[:, j], color='tab:orange', lw=1.2, label='ERC-objective (SML-ERC-HRP)')
    ax.set_title(feat)
    ax.set_xlabel('Walk-forward window (0-78)')
    ax.set_ylabel('Fitted weight (theta**2)')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.04))
fig.suptitle('SML fitted per-feature weights across 79 walk-forward windows: variance vs ERC objective', y=1.09)
plt.tight_layout()
# Save the figure inside the Colab runtime.
plt.savefig('/content/weight_diagnostic_2026-08-11.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved to /content/weight_diagnostic_2026-08-11.png')

### 13.2 Similarity to PCA allocations

* Recompute training features and HRP allocations using stored feature weights, then measure Euclidean distance from cached PCA allocations.
* No metric optimization is repeated.
* Closer allocations indicate similarity, not proof that ERC learns no information.


In [ ]:
# Reconstruct allocations from fitted feature weights, without new metric optimization.
# Compare portfolio distances to PCA; similarity does not establish a mechanism.

from scipy import stats
n_win = len(windows)
assert len(learned_weights_log) == n_win == len(erc_weights_log) == len(pca_weights_per_window)
dist_sml_pca = []
dist_erc_pca = []
for k, (ts, te, vs, ve) in enumerate(windows):
    tr = returns.loc[ts:te]
    feats_df = build_asset_features(tr)
    d_sml = weighted_feature_distance(feats_df, learned_weights_log[k])
    d_erc = weighted_feature_distance(feats_df, erc_weights_log[k])
    w_sml = hrp_weights_from_dist(tr, d_sml)[SYMBOLS].values
    w_erc = hrp_weights_from_dist(tr, d_erc)[SYMBOLS].values
    # Explicit SYMBOLS indexing aligns all three portfolio arrays before subtraction.
    w_pca = pca_weights_per_window[k][SYMBOLS].values
    dist_sml_pca.append(np.sqrt(((w_sml - w_pca)**2).sum()))
    dist_erc_pca.append(np.sqrt(((w_erc - w_pca)**2).sum()))
dist_sml_pca = np.array(dist_sml_pca)
dist_erc_pca = np.array(dist_erc_pca)
print('n windows:', n_win)
print('SML-HRP -> PCA-HRP dist: mean=', dist_sml_pca.mean(), 'median=', np.median(dist_sml_pca), 'std=', dist_sml_pca.std(ddof=1))
print('SML-ERC-HRP -> PCA-HRP dist: mean=', dist_erc_pca.mean(), 'median=', np.median(dist_erc_pca), 'std=', dist_erc_pca.std(ddof=1))
# Test paired per-window allocation distances, not return differences.
t_res = stats.ttest_rel(dist_sml_pca, dist_erc_pca)
w_res = stats.wilcoxon(dist_sml_pca, dist_erc_pca, alternative='greater')
n_erc_closer = int((dist_erc_pca < dist_sml_pca).sum())
print('Paired t-test (sml_dist - erc_dist): t=', t_res.statistic, 'p=', t_res.pvalue)
print('Wilcoxon H1 sml_dist > erc_dist: stat=', w_res.statistic, 'p=', w_res.pvalue)
print('Windows ERC closer to PCA-HRP:', n_erc_closer, '/', n_win)

### 13.3 Reconstructing portfolio returns

* Rebuild returns from saved feature weights and print pooled Sharpes beside historical references.
* Compare full return series for stronger verification.


In [ ]:
# Reconstruct daily portfolio returns from stored feature weights.

verify_sml_rets = []
verify_erc_rets = []
for k, (ts, te, vs, ve) in enumerate(windows):
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    feats_df = build_asset_features(tr)
    d_sml = weighted_feature_distance(feats_df, learned_weights_log[k])
    d_erc = weighted_feature_distance(feats_df, erc_weights_log[k])
    w_sml = hrp_weights_from_dist(tr, d_sml)
    w_erc = hrp_weights_from_dist(tr, d_erc)
    verify_sml_rets.append((tst @ w_sml[SYMBOLS]).rename('SML-HRP'))
    verify_erc_rets.append((tst @ w_erc[SYMBOLS]).rename('SML-ERC-HRP'))
# Concatenate reconstructed chunks; agreement in a single Sharpe statistic
# would not prove every daily return matches the original series.
verify_sml_series = pd.concat(verify_sml_rets)
verify_erc_series = pd.concat(verify_erc_rets)
print('Recomputed SML-HRP Sharpe:', sharpe_lo2002(verify_sml_series))
print('Recomputed SML-ERC-HRP Sharpe:', sharpe_lo2002(verify_erc_series))

### 13.4 Live descriptive inputs for future DSR validation
Section 17 supplies a conditional DSR calculation; a complete historical-search correction is not claimed. The following cells report current moments only.


In [ ]:
# Collect excess kurtosis and other moments

import scipy.stats as sps
window_sharpes = np.array([sharpe_lo2002(c) for c in erc_chunks])
T_windows = len(window_sharpes)
mean_ws = window_sharpes.mean()
std_ws = window_sharpes.std(ddof=1)
skew_windows = sps.skew(window_sharpes, bias=False)
# fisher=True reports EXCESS kurtosis. A later formula requiring ordinary
kurt_windows = sps.kurtosis(window_sharpes, fisher=True, bias=False)
skew_daily = erc_series.skew()
kurt_daily = erc_series.kurt()
T_daily = len(erc_series)
pooled_sharpe = sharpe_lo2002(erc_series)
print('DSR_STATS', 'T_windows', T_windows, 'mean_ws', mean_ws, 'std_ws', std_ws)
print('DSR_STATS', 'skew_windows', skew_windows, 'kurt_windows_excess', kurt_windows)
print('DSR_STATS', 'T_daily', T_daily, 'skew_daily', skew_daily, 'kurt_daily_excess', kurt_daily)
print('DSR_STATS', 'pooled_sharpe', pooled_sharpe)
print('DSR_STATS', 'window_sharpes_list', [round(float(x),6) for x in window_sharpes])

In [ ]:
# Live descriptive inputs only: a validated DSR requires a complete trial registry
# and consistent return frequency, moments, dependence treatment and selection model.
dsr_inputs = {"n_daily": len(erc_series), "daily_mean": erc_series.mean(),
              "daily_std": erc_series.std(ddof=1), "daily_sharpe_unadjusted": erc_series.mean()/erc_series.std(ddof=1),
              "daily_skew": erc_series.skew(), "daily_kurtosis_ordinary": erc_series.kurt()+3,
              "pooled_adjusted_sharpe": sharpe_lo2002(erc_series)}
print(pd.Series(dsr_inputs))
print("DSR not reported: complete selection-trial registry and dependence specification required.")


## 14. Cross-method summaries

* Report mean window Sharpe across methods and calendar regimes.
* Run after all experiment sections.
* These summaries are not pooled full-series Sharpes. The additional report below collects selected pooled metrics.


In [ ]:
# Summarize mean window Sharpe overall and by regime after all experiments.

pregfc_idx = [i for i, r in enumerate(regime_labels) if r == 'Pre-GFC']
gfc_idx = [i for i, r in enumerate(regime_labels) if r == 'GFC']
postgfc_idx = [i for i, r in enumerate(regime_labels) if r == 'Post-GFC-to-COVID']
covid_idx = [i for i, r in enumerate(regime_labels) if r == 'COVID+']
regime_idx_map = {'Pre-GFC': pregfc_idx, 'GFC': gfc_idx, 'Post-GFC-to-COVID': postgfc_idx, 'COVID+': covid_idx}
# Select the regime’s chunks and average their individually computed Sharpes.
# An empty subset would produce an undefined mean; no guard is applied here.
def rm(chunks, idx): return quarterly_sharpes([chunks[i] for i in idx]).mean()
results = {}
results['base_chunks'] = m(base_chunks)
results['ae_chunks'] = m(ae_chunks)
results['sml_chunks'] = m(sml_chunks)
results['pca_chunks'] = m(pca_chunks)
results['erc_chunks'] = m(erc_chunks)
results['cmaes_chunks'] = m(cmaes_chunks)
results['shrink_chunks_best'] = m(shrink_chunks_best)
results['ae_erc_chunks'] = m(ae_erc_chunks)
results['corr_herc_chunks'] = m(corr_herc_chunks)
results['pca_herc_chunks'] = m(pca_herc_chunks)
results['sml_herc_chunks'] = m(sml_herc_chunks)
regime_results = {(label, regime): rm(chunks, idx) for label, chunks in [('base_chunks', base_chunks), ('ae_chunks', ae_chunks), ('sml_chunks', sml_chunks), ('pca_chunks', pca_chunks), ('corr_herc_chunks', corr_herc_chunks), ('pca_herc_chunks', pca_herc_chunks), ('sml_herc_chunks', sml_herc_chunks)] for regime, idx in regime_idx_map.items()}
ae_erc_gfc = rm(ae_erc_chunks, gfc_idx)
pca_gfc = rm(pca_chunks, gfc_idx)
print('FULL79', results)
print()
print('REGIME', regime_results)
print()
print('AEERC_GFC', ae_erc_gfc, 'PCA_GFC', pca_gfc)

### 14.1 Mean window Sharpe for SML variants
* Summarize the current variance, shrinkage, ERC and CMA-ES variants against PCA.
* All values are calculated from the current fitted returns.


In [ ]:
# Current mean window Sharpes and differences; no historical constants.
for name, chunks in {"PCA": pca_chunks, "SML": sml_chunks, "Shrink": shrink_results[0.005]["chunks"], "ERC": erc_chunks, "CMA-ES": cmaes_chunks}.items():
    print(name, "mean window Sharpe", m(chunks), "difference versus PCA", m(chunks)-m(pca_chunks))


In [ ]:
# Plot candidate-minus-PCA Sharpe for each matching test window.
# Positive values favour the candidate; dates are test-window starts.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

window_dates = [w[2] for w in windows]

def per_window_sharpe_diff(candidate_chunks, baseline_chunks):
    # Pair by list position. This assumes all methods completed the same windows
    # in the same order; the helper does not validate dates or list lengths.
    return np.array([
        sharpe_lo2002(candidate_chunks[i]) - sharpe_lo2002(baseline_chunks[i])
        for i in range(len(baseline_chunks))
    ])

diff_df = pd.DataFrame({
    'SML-HRP - PCA-HRP': per_window_sharpe_diff(sml_chunks, pca_chunks),
    'SML-Shrink-HRP - PCA-HRP': per_window_sharpe_diff(shrink_results[0.005]['chunks'], pca_chunks),
    'SML-ERC-HRP - PCA-HRP': per_window_sharpe_diff(erc_chunks, pca_chunks),
    'SML-CMAES-HRP - PCA-HRP': per_window_sharpe_diff(cmaes_chunks, pca_chunks),
}, index=window_dates)

fig, ax = plt.subplots(figsize=(11, 5.5))

diff_df.plot(ax=ax, linewidth=1.4)

ax.axhline(0, color='black', linewidth=1, alpha=0.7)
ax.set_title('Window-level Sharpe differences versus PCA-HRP')
ax.set_xlabel('Test-window start date')
ax.set_ylabel('Candidate Sharpe minus PCA-HRP Sharpe')
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.savefig('fig_phase2_window_sharpe_differences.png', dpi=150, bbox_inches='tight')
plt.show()

print(diff_df.describe().round(4))

In [ ]:
# Display pooled performance for all fitted methods in one table.
# Mean window Sharpe is a separate summary and is not the pooled Sharpe.
method_series = {
    'Correlation-HRP': base_series,
    'PCA-HRP': pca_series,
    'AE-HRP': ae_series,
    'SML-HRP': sml_series,
    'SML-Shrink': shrink_results[best_lam]['series'],
    'SML-ERC': erc_series,
    'SML-CMAES': cmaes_series,
    'AE-ERC': ae_erc_series,
    'Correlation custom allocator': corr_herc_series,
    'PCA custom allocator': pca_herc_series,
    'SML custom allocator': sml_herc_series,
    'Regime shrinkage (exploratory)': rshrink_series,
}
# Compute each statistic from the full concatenated daily return series.
# Transpose the dictionary-based table so methods become rows and metrics columns.
pooled_summary = pd.DataFrame({
    name: {'Pooled Sharpe': sharpe_lo2002(series),
           'Max drawdown': max_drawdown(series),
           'CEQ (gamma=1)': ceq(series, 1)}
    for name, series in method_series.items()
}).T
print(pooled_summary.round(4).to_string())


## 15. Consolidated inference and daily costs

* This cell refreshes comparisons using fitted chunks already in memory; it does not refit models.
* Run the preceding experiment sections first on a fresh runtime. On an existing fitted runtime, run this cell alone to refresh inference and costs.

* Read the output in this order: learned methods versus correlation and PCA; SML variants and direct ERC-versus-variance comparisons; HERC-style allocator; AE-ERC; multiplicity corrections; block-length sensitivity; chronological halves and daily costs; descriptive DSR inputs.


In [ ]:
# Refresh inference from existing fitted chunks. No models are refitted.
def bootstrap_sharpe_diff(chunks_a, chunks_b, n_boot=5000, seed=42, block_length=4):
    """Circular block bootstrap of paired window Sharpes; null-centred two-sided test.

    Default block length is four quarterly windows, with sensitivity reported below.
    This preserves local dependence within sampled blocks, not all dependence.
    Short regime samples remain exploratory. CI is a basic bootstrap interval.
    """
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = quarterly_sharpes(chunks_a) - quarterly_sharpes(chunks_b)
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window Sharpe; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)


# Refresh original code cell 20
# The reported delta and interval summarize paired per-window differences.
# The table below instead recomputes metrics on concatenated daily returns.
# Neither the raw bootstrap tail value nor this cell adjusts for all model searches.

# Compare average paired window Sharpes, then display pooled performance.
# The two Sharpe summaries need not have the same difference.

obs, lo, hi, p = bootstrap_sharpe_diff(ae_chunks, base_chunks)
print("=== AE-HRP vs Correlation-HRP (champion proxy) ===")
print(f"Delta Sharpe:  {obs:+.4f}")
print(f"95% CI:        [{lo:+.4f}, {hi:+.4f}]")
print(f"p-value:       {p:.4f}")
print()
print("Summary table")
print("-" * 60)
summary = pd.DataFrame({
    'Sharpe':          [sharpe_lo2002(base_series), sharpe_lo2002(ae_series)],
    'MaxDD':           [max_drawdown(base_series),   max_drawdown(ae_series)],
    'CEQ (g=1)':       [ceq(base_series, 1),         ceq(ae_series, 1)],
    'CEQ (g=3)':       [ceq(base_series, 3),         ceq(ae_series, 3)],
    'Avg Turnover':    [np.mean(base_turnover),       np.mean(ae_turnover)],
}, index=['Correlation-HRP (champion proxy)', 'AE-Embedding-HRP (learned)'])
print(summary.round(4).to_string())

# Refresh original code cell 23
# Select the same calendar-regime windows for both candidates.

regime_bounds = [
    ('Pre-GFC',           pd.Timestamp('1900-01-01'), pd.Timestamp('2008-01-01')),
    ('GFC',               pd.Timestamp('2008-01-01'), pd.Timestamp('2010-01-01')),
    ('Post-GFC-to-COVID', pd.Timestamp('2010-01-01'), pd.Timestamp('2020-01-01')),
    ('COVID+',            pd.Timestamp('2020-01-01'), pd.Timestamp('2100-01-01')),
]
def regime_of(test_start):
    for name, lo, hi in regime_bounds:
        if lo <= test_start < hi:
            return name
    return 'Unknown'

regime_labels = [regime_of(vs) for (ts, te, vs, ve) in windows]

rows = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    base_c = [base_chunks[i] for i in idx]
    ae_c   = [ae_chunks[i] for i in idx]
    base_r = pd.concat(base_c)
    ae_r   = pd.concat(ae_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(ae_c, base_c)
    rows.append({
        'Regime': name,
        'N windows': len(idx),
        'Corr-HRP Sharpe': sharpe_lo2002(base_r),
        'AE-HRP Sharpe': sharpe_lo2002(ae_r),
        'Delta Sharpe': obs,
        '95% CI lo': lo_ci,
        '95% CI hi': hi_ci,
        'p-value': p,
    })

regime_table = pd.DataFrame(rows).set_index('Regime')
print("=== Regime sub-split: AE-Embedding-HRP vs Correlation-HRP ===")
print(regime_table.round(4).to_string())

# Refresh original code cell 27
# Feature-weight logs describe the fitted distance. Portfolio metrics describe
# the HRP holdings obtained from that distance, a different set of weights.

# Report SML versus correlation and pooled portfolio metrics.
# Normalize feature weights per window before interpreting relative importance.

obs, lo, hi, p = bootstrap_sharpe_diff(sml_chunks, base_chunks)
print("=== SML-HRP vs Correlation-HRP (champion proxy) ===")
print(f"Delta Sharpe:  {obs:+.4f}")
print(f"95% CI:        [{lo:+.4f}, {hi:+.4f}]")
print(f"p-value:       {p:.4f}")
print()
# These printed means are RAW weights. Multiplying all feature weights by the
# same positive constant rescales distances without changing the HRP ordering,
# so raw means are not normalized feature-importance estimates.
avg_w = np.mean(learned_weights_log, axis=0)
print("Avg learned feature weights across windows:")
for name, val in zip(FEATURE_NAMES, avg_w):
    print(f"  {name}: {val:.3f}")

print()
summary2 = pd.DataFrame({
    'Sharpe':       [sharpe_lo2002(base_series), sharpe_lo2002(ae_series), sharpe_lo2002(sml_series)],
    'MaxDD':        [max_drawdown(base_series), max_drawdown(ae_series), max_drawdown(sml_series)],
    'CEQ (g=1)':    [ceq(base_series, 1), ceq(ae_series, 1), ceq(sml_series, 1)],
    'CEQ (g=3)':    [ceq(base_series, 3), ceq(ae_series, 3), ceq(sml_series, 3)],
    'Avg Turnover': [np.mean(base_turnover), np.mean(ae_turnover), np.mean(sml_turnover)],
}, index=['Correlation-HRP (champion proxy)', 'AE-Embedding-HRP', 'SML-HRP (Experiment 2)'])
print(summary2.round(4).to_string())

# Refresh original code cell 29
# Compare SML with correlation separately in each calendar regime.

rows_sml = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    base_c = [base_chunks[i] for i in idx]
    sml_c = [sml_chunks[i] for i in idx]
    base_r = pd.concat(base_c)
    sml_r = pd.concat(sml_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(sml_c, base_c)
    rows_sml.append({
        'Regime': name,
        'N windows': len(idx),
        'Corr-HRP Sharpe': sharpe_lo2002(base_r),
        'SML-HRP Sharpe': sharpe_lo2002(sml_r),
        'Delta Sharpe': obs,
        '95% CI lo': lo_ci,
        '95% CI hi': hi_ci,
        'p-value': p,
    })

regime_table_sml = pd.DataFrame(rows_sml).set_index('Regime')
print("=== Regime sub-split: SML-HRP vs Correlation-HRP ===")
print(regime_table_sml.round(4).to_string())

# Refresh original code cell 30
# Correct the current regime comparisons as one family using Holm.
# Keep full precision; this does not validate the underlying bootstrap test.
from statsmodels.stats.multitest import multipletests

comparison_tables_a = {'AE': regime_table, 'SML': regime_table_sml}
pval_labels_a, pvals_a = [], []
for method, table in comparison_tables_a.items():
    for regime, value in table['p-value'].items():
        pval_labels_a.append(f'{method} [{regime}]')
        pvals_a.append(value)

pvals_a = np.asarray(pvals_a, dtype=float)
if pvals_a.size == 0 or not np.all(np.isfinite(pvals_a) & (pvals_a >= 0) & (pvals_a <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')
# Apply Holm at family-wise alpha=0.05 to this cell’s collected comparisons.
reject_a, pvals_holm_a, _, _ = multipletests(pvals_a, alpha=0.05, method='holm')
holm_df_a = pd.DataFrame(
    {'p_raw': pvals_a, 'p_holm': pvals_holm_a, 'survives_0.05': reject_a},
    index=pval_labels_a,
)
print(f'=== Holm correction: learned versus correlation, {len(pvals_a)} tests ===')
print(holm_df_a.round(4).to_string())


# Refresh original code cell 34
# Positive deltas favour the learned method, since each call is candidate minus PCA.
# A confidence interval spanning zero does not establish equivalence.

# Compare AE and SML with PCA using mean paired window Sharpe differences.

obs_ae, lo_ae, hi_ae, p_ae = bootstrap_sharpe_diff(ae_chunks, pca_chunks)
obs_sml, lo_sml, hi_sml, p_sml = bootstrap_sharpe_diff(sml_chunks, pca_chunks)

print("=== AE-Embedding-HRP vs PCA-HRP (Phase 1 champion) ===")
print(f"Delta Sharpe: {obs_ae:+.4f}  95% CI [{lo_ae:+.4f}, {hi_ae:+.4f}]  p-value: {p_ae:.4f}")

print("=== SML-HRP vs PCA-HRP (Phase 1 champion) ===")
print(f"Delta Sharpe: {obs_sml:+.4f}  95% CI [{lo_sml:+.4f}, {hi_sml:+.4f}]  p-value: {p_sml:.4f}")

# Refresh original code cell 35
# Each row uses pooled daily returns; average turnover uses successive target
# changes and therefore normally contains one fewer observation than the window count.

# Display pooled performance across the four main methods.

summary_pca = pd.DataFrame({
    'Sharpe': [sharpe_lo2002(pca_series), sharpe_lo2002(base_series), sharpe_lo2002(ae_series), sharpe_lo2002(sml_series)],
    'MaxDD': [max_drawdown(pca_series), max_drawdown(base_series), max_drawdown(ae_series), max_drawdown(sml_series)],
    'CEQ (g=1)': [ceq(pca_series, 1), ceq(base_series, 1), ceq(ae_series, 1), ceq(sml_series, 1)],
    'Avg Turnover': [np.mean(pca_turnover), np.mean(base_turnover), np.mean(ae_turnover), np.mean(sml_turnover)],
}, index=['PCA-HRP (Phase 1 champion)', 'Correlation-HRP', 'AE-Embedding-HRP', 'SML-HRP'])

print("=== Summary vs PCA-HRP (Phase 1 champion) baseline, 79 windows ===")
print(summary_pca.round(4).to_string())

# Refresh original code cell 36
# Report pooled regime metrics and paired window Sharpe differences versus PCA.

rows_ae_pca = []
rows_sml_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    ae_c = [ae_chunks[i] for i in idx]
    sml_c = [sml_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    ae_r = pd.concat(ae_c)
    sml_r = pd.concat(sml_c)
    obs_a, lo_a, hi_a, p_a = bootstrap_sharpe_diff(ae_c, pca_c)
    obs_s, lo_s, hi_s, p_s = bootstrap_sharpe_diff(sml_c, pca_c)
    row_a = {'Regime': name, 'N windows': len(idx)}
    row_a['PCA-HRP Sharpe'] = sharpe_lo2002(pca_r)
    row_a['AE-HRP Sharpe'] = sharpe_lo2002(ae_r)
    row_a['Delta Sharpe'] = obs_a
    row_a['95% CI lo'] = lo_a
    row_a['95% CI hi'] = hi_a
    row_a['p-value'] = p_a
    rows_ae_pca.append(row_a)
    row_s = {'Regime': name, 'N windows': len(idx)}
    row_s['PCA-HRP Sharpe'] = sharpe_lo2002(pca_r)
    row_s['SML-HRP Sharpe'] = sharpe_lo2002(sml_r)
    row_s['Delta Sharpe'] = obs_s
    row_s['95% CI lo'] = lo_s
    row_s['95% CI hi'] = hi_s
    row_s['p-value'] = p_s
    rows_sml_pca.append(row_s)

regime_table_ae_pca = pd.DataFrame(rows_ae_pca).set_index('Regime')
regime_table_sml_pca = pd.DataFrame(rows_sml_pca).set_index('Regime')

print("=== Regime sub-split: AE-Embedding-HRP vs PCA-HRP ===")
print(regime_table_ae_pca.round(4).to_string())

print("=== Regime sub-split: SML-HRP vs PCA-HRP ===")
print(regime_table_sml_pca.round(4).to_string())

# Refresh original code cell 37
# Correct the current regime comparisons as one family using Holm.
# Keep full precision; this does not validate the underlying bootstrap test.
from statsmodels.stats.multitest import multipletests

comparison_tables_b = {'AE': regime_table_ae_pca, 'SML': regime_table_sml_pca}
pval_labels_b, pvals_b = [], []
for method, table in comparison_tables_b.items():
    for regime, value in table['p-value'].items():
        pval_labels_b.append(f'{method} [{regime}]')
        pvals_b.append(value)

pvals_b = np.asarray(pvals_b, dtype=float)
if pvals_b.size == 0 or not np.all(np.isfinite(pvals_b) & (pvals_b >= 0) & (pvals_b <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')
# Apply Holm at family-wise alpha=0.05 to this cell’s collected comparisons.
reject_b, pvals_holm_b, _, _ = multipletests(pvals_b, alpha=0.05, method='holm')
holm_df_b = pd.DataFrame(
    {'p_raw': pvals_b, 'p_holm': pvals_holm_b, 'survives_0.05': reject_b},
    index=pval_labels_b,
)
print(f'=== Holm correction: learned versus PCA, {len(pvals_b)} tests ===')
print(holm_df_b.round(4).to_string())


# Refresh original code cell 42
# The helper resamples circular blocks of consecutive paired
# window Sharpe differences using the shared block-length setting.
# The comparisons for different lambda values are not corrected together here.

# Compare each shrinkage setting with both benchmarks. These are raw comparisons.

print("=== SML-Shrink-HRP significance vs Correlation-HRP and PCA-HRP (block bootstrap) ===")
sig_rows = []
for lam in lam_grid:
    chunks_lam = shrink_results[lam]['chunks']
    obs_c, lo_c, hi_c, p_c = bootstrap_sharpe_diff(chunks_lam, base_chunks)
    obs_p, lo_p, hi_p, p_p = bootstrap_sharpe_diff(chunks_lam, pca_chunks)
    sig_rows.append((lam, obs_c, lo_c, hi_c, p_c, obs_p, lo_p, hi_p, p_p))
    print(f"lam={lam} vs Corr dSharpe={obs_c:.4f} CI_lo={lo_c:.4f} CI_hi={hi_c:.4f} p={p_c:.4f} | vs PCA dSharpe={obs_p:.4f} CI_lo={lo_p:.4f} CI_hi={hi_p:.4f} p={p_p:.4f}")

# Refresh original code cell 43
# Use the manually selected lambda=0.005 for regime comparisons.
# The variable best_lam does not implement an independent selection rule.

# This constant is chosen manually. It does not estimate lambda on a separate
# validation set or prove that 0.005 is optimal.
best_lam = 0.005
shrink_chunks_best = shrink_results[best_lam]['chunks']
rows_shrink_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    shrink_c = [shrink_chunks_best[i] for i in idx]
    pca_r = pd.concat(pca_c)
    shrink_r = pd.concat(shrink_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(shrink_c, pca_c)
    rows_shrink_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-Shrink-HRP Sharpe': sharpe_lo2002(shrink_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p})

regime_table_shrink_pca = pd.DataFrame(rows_shrink_pca).set_index('Regime')
print(f"=== Regime sub-split: SML-Shrink-HRP (lam={best_lam}) vs PCA-HRP ===")
print(regime_table_shrink_pca.round(4).to_string())

# Refresh original code cell 47
# Both calls evaluate the same ERC candidate against different comparators.
# These are raw separate comparisons, not a corrected two-test family.

# Compare SML-ERC with correlation and PCA using the same paired-window procedure.

obs_erc_corr, lo_erc_corr, hi_erc_corr, p_erc_corr = bootstrap_sharpe_diff(erc_chunks, base_chunks)
obs_erc_pca, lo_erc_pca, hi_erc_pca, p_erc_pca = bootstrap_sharpe_diff(erc_chunks, pca_chunks)
print(f'SML-ERC-HRP vs Correlation-HRP: dSharpe={obs_erc_corr:+.4f} CI=[{lo_erc_corr:+.4f},{hi_erc_corr:+.4f}] p={p_erc_corr:.4f}')
print(f'SML-ERC-HRP vs PCA-HRP: dSharpe={obs_erc_pca:+.4f} CI=[{lo_erc_pca:+.4f},{hi_erc_pca:+.4f}] p={p_erc_pca:.4f}')

# Refresh original code cell 48
# Compare SML-ERC with PCA inside GFC and COVID+ calendar windows.

gfc_idx = [i for i, r in enumerate(regime_labels) if r == 'GFC']
covid_idx = [i for i, r in enumerate(regime_labels) if r == 'COVID+']
gfc_erc = [erc_chunks[i] for i in gfc_idx]
gfc_pca = [pca_chunks[i] for i in gfc_idx]
covid_erc = [erc_chunks[i] for i in covid_idx]
covid_pca = [pca_chunks[i] for i in covid_idx]
obs_g, lo_g, hi_g, p_g = bootstrap_sharpe_diff(gfc_erc, gfc_pca)
obs_c, lo_c, hi_c, p_c = bootstrap_sharpe_diff(covid_erc, covid_pca)
print(f"GFC: dSharpe={obs_g:+.4f} CI=[{lo_g:+.4f},{hi_g:+.4f}] p={p_g:.4f}")
print(f"COVID+: dSharpe={obs_c:+.4f} CI=[{lo_c:+.4f},{hi_c:+.4f}] p={p_c:.4f}")

# Refresh original code cell 50
gfc_idx = [i for i, r in enumerate(regime_labels) if r == 'GFC']
covid_idx = [i for i, r in enumerate(regime_labels) if r == 'COVID+']
overall = bootstrap_sharpe_diff(erc_chunks, sml_chunks, n_boot=5000, seed=42)
gfc_erc = [erc_chunks[i] for i in gfc_idx]
gfc_sml = [sml_chunks[i] for i in gfc_idx]
gfc = bootstrap_sharpe_diff(gfc_erc, gfc_sml, n_boot=5000, seed=42)
covid_erc = [erc_chunks[i] for i in covid_idx]
covid_sml = [sml_chunks[i] for i in covid_idx]
covid = bootstrap_sharpe_diff(covid_erc, covid_sml, n_boot=5000, seed=42)
print('=== SML-ERC-HRP vs SML-HRP direct paired block-bootstrap ===')
print('Overall (n=79): mean_diff, ci_lo, ci_hi, p =', overall)
print('GFC (n=8):      mean_diff, ci_lo, ci_hi, p =', gfc)
print('COVID+ (n=25):  mean_diff, ci_lo, ci_hi, p =', covid)

# Refresh original code cell 54
# Compare CMA-ES with correlation and PCA using paired window differences.

obs_cmaes_corr, lo_cmaes_corr, hi_cmaes_corr, p_cmaes_corr = bootstrap_sharpe_diff(cmaes_chunks, base_chunks)
obs_cmaes_pca, lo_cmaes_pca, hi_cmaes_pca, p_cmaes_pca = bootstrap_sharpe_diff(cmaes_chunks, pca_chunks)
print(f'SML-CMAES-HRP vs Correlation-HRP: dSharpe={obs_cmaes_corr:+.4f} CI=[{lo_cmaes_corr:+.4f},{hi_cmaes_corr:+.4f}] p={p_cmaes_corr:.4f}')
print(f'SML-CMAES-HRP vs PCA-HRP: dSharpe={obs_cmaes_pca:+.4f} CI=[{lo_cmaes_pca:+.4f},{hi_cmaes_pca:+.4f}] p={p_cmaes_pca:.4f}')

# Refresh original code cell 55
# Repeat the CMA-ES versus PCA comparison within market regimes.

rows_cmaes_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_chunks[i] for i in idx]
    cmaes_c = [cmaes_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    cmaes_r = pd.concat(cmaes_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(cmaes_c, pca_c)
    rows_cmaes_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-CMAES-HRP Sharpe': sharpe_lo2002(cmaes_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p})
regime_table_cmaes_pca = pd.DataFrame(rows_cmaes_pca).set_index('Regime')
print('=== Regime sub-split: SML-CMAES-HRP vs PCA-HRP ===')
print(regime_table_cmaes_pca.round(4).to_string())

# Refresh original code cell 57
# Compare resampling seeds using current fitted returns.
for seed in (42, 999):
    result = bootstrap_sharpe_diff(cmaes_chunks, pca_chunks, seed=seed)
    print(f"CMA-ES versus PCA, seed={seed}: delta, CI low, CI high, p = {result}")


# Refresh original code cell 62
# Compare SML and PCA under the modified allocator by calendar regime.

rows_herc = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0:
        continue
    pca_c = [pca_herc_chunks[i] for i in idx]
    sml_c = [sml_herc_chunks[i] for i in idx]
    pca_r = pd.concat(pca_c)
    sml_r = pd.concat(sml_c)
    obs, lo_ci, hi_ci, p = bootstrap_sharpe_diff(sml_c, pca_c)
    rows_herc.append({'Regime': name, 'N windows': len(idx), 'PCA-HERC Sharpe': sharpe_lo2002(pca_r), 'SML-HERC Sharpe': sharpe_lo2002(sml_r), 'Delta Sharpe': obs, '95% CI lo': lo_ci, '95% CI hi': hi_ci, 'p-value': p})
regime_table_herc = pd.DataFrame(rows_herc).set_index('Regime')
print("=== Regime sub-split: SML-HERC vs PCA-HERC (all four regimes) ===")
print(regime_table_herc.round(4).to_string())

# Refresh original code cell 63
# Holm-bonferroni correction

from statsmodels.stats.multitest import multipletests

comparison_tables_c = {'SML variant versus PCA variant': regime_table_herc}
pval_labels_c, pvals_c = [], []
for method, table in comparison_tables_c.items():
    for regime, value in table['p-value'].items():
        pval_labels_c.append(f'{method} [{regime}]')
        pvals_c.append(value)

pvals_c = np.asarray(pvals_c, dtype=float)
if pvals_c.size == 0 or not np.all(np.isfinite(pvals_c) & (pvals_c >= 0) & (pvals_c <= 1)):
    raise ValueError('Holm correction requires finite p-values between 0 and 1.')

reject_c, pvals_holm_c, _, _ = multipletests(pvals_c, alpha=0.05, method='holm')
holm_df_c = pd.DataFrame(
    {'p_raw': pvals_c, 'p_holm': pvals_holm_c, 'survives_0.05': reject_c},
    index=pval_labels_c,
)
print(f'=== Holm correction: modified allocator, {len(pvals_c)} tests ===')
print(holm_df_c.round(4).to_string())


# Refresh original code cell 68
# m() averages one Sharpe per chunk with equal window weight.
# Compare AE-ERC with PCA overall and in GFC

ae_erc_gfc = [ae_erc_chunks[i] for i in gfc_idx]
pca_gfc = [pca_chunks[i] for i in gfc_idx]
print('MEAN_AE_ERC_FULL', m(ae_erc_chunks))
print('MEAN_PCA_FULL', m(pca_chunks))
print('MEAN_AE_ERC_GFC', m(ae_erc_gfc))
print('MEAN_PCA_GFC', m(pca_gfc))
print('check overall diff', m(ae_erc_chunks)-m(pca_chunks))
print('check gfc diff', m(ae_erc_gfc)-m(pca_gfc))
obs,lo,hi,p = bootstrap_sharpe_diff(ae_erc_chunks, pca_chunks)
print('bootstrap overall', obs, lo, hi, p)
obs2,lo2,hi2,p2 = bootstrap_sharpe_diff(ae_erc_gfc, pca_gfc)
print('bootstrap gfc', obs2, lo2, hi2, p2)


obs_herc, lo_herc, hi_herc, p_herc = bootstrap_sharpe_diff(sml_herc_chunks, pca_herc_chunks)
print(f'SML-HERC vs PCA-HERC: dSharpe={obs_herc:+.4f} CI=[{lo_herc:+.4f},{hi_herc:+.4f}] p={p_herc:.4f}')

obs_rc, lo_rc, hi_rc, p_rc = bootstrap_sharpe_diff(rshrink_chunks, base_chunks)
obs_rp, lo_rp, hi_rp, p_rp = bootstrap_sharpe_diff(rshrink_chunks, pca_chunks)
print(f"SML-Regime-Shrink-HRP vs Correlation-HRP: dSharpe={obs_rc:+.4f} CI=[{lo_rc:+.4f},{hi_rc:+.4f}] p={p_rc:.4f}")
print(f"SML-Regime-Shrink-HRP vs PCA-HRP: dSharpe={obs_rp:+.4f} CI=[{lo_rp:+.4f},{hi_rp:+.4f}] p={p_rp:.4f}")
rows_rshrink_pca = []
for name, _, _ in regime_bounds:
    # Select list positions using the test-window start-date label.
    # A window crossing a calendar boundary stays entirely in its starting regime.
    idx = [i for i, r in enumerate(regime_labels) if r == name]
    if len(idx) == 0: continue
    pca_c = [pca_chunks[i] for i in idx]; rshrink_c = [rshrink_chunks[i] for i in idx]; lams_in = [rshrink_lams[i] for i in idx]
    pca_r = pd.concat(pca_c); rshrink_r = pd.concat(rshrink_c)
    obs, lo, hi, p = bootstrap_sharpe_diff(rshrink_c, pca_c)
    rows_rshrink_pca.append({'Regime': name, 'N': len(idx), 'PCA-HRP Sharpe': sharpe_lo2002(pca_r), 'SML-Regime-Shrink-HRP Sharpe': sharpe_lo2002(rshrink_r), 'Delta Sharpe': obs, '95% CI lo': lo, '95% CI hi': hi, 'p-value': p, 'N_high_lam': sum(1 for l in lams_in if l==LAM_HIGH)})
regime_table_rshrink_pca = pd.DataFrame(rows_rshrink_pca).set_index('Regime')
print("=== Regime sub-split: SML-Regime-Shrink-HRP vs PCA-HRP ===")
print(regime_table_rshrink_pca.round(4).to_string())
print('GFC lam values:', [rshrink_lams[i] for i in gfc_idx])
print('COVID+ lam values:', [rshrink_lams[i] for i in covid_idx])

# Full-sample comparisons with PCA: one declared family, Holm-adjusted.
from statsmodels.stats.multitest import multipletests
comparison_chunks = {'AE': ae_chunks, 'SML': sml_chunks,
    'Shrink': shrink_results[0.005]['chunks'], 'ERC': erc_chunks,
    'CMA-ES': cmaes_chunks, 'AE-ERC': ae_erc_chunks, 'Regime shrinkage': rshrink_chunks}
inference_rows = []
for name, chunks in comparison_chunks.items():
    delta, low, high, p = bootstrap_sharpe_diff(chunks, pca_chunks)
    inference_rows.append([name, delta, low, high, p])
updated_inference = pd.DataFrame(inference_rows, columns=['Method', 'Delta', 'CI low', 'CI high', 'p raw'])
updated_inference['p Holm (7 comparisons)'] = multipletests(updated_inference['p raw'], method='holm')[1]
print(updated_inference.to_string(index=False))
print('Direct ERC-versus-SML: Holm across overall, GFC and COVID+')
print(multipletests([overall[3], gfc[3], covid[3]], method='holm')[1])
print('Block-length sensitivity: full-sample candidate versus PCA')
for length in (2, 4, 8):
    for name, chunks in comparison_chunks.items():
        print(length, name, bootstrap_sharpe_diff(chunks, pca_chunks, block_length=length))

# Compare chronological halves using pooled return metrics.
# Costs below are applied to daily self-financing trades.

# With 79 windows this gives 39 in the first half and 40 in the second.
# Concatenation preserves full daily series rather than averaging window Sharpes.
n_first = len(windows) // 2
first_erc = pd.concat(erc_chunks[:n_first])
second_erc = pd.concat(erc_chunks[n_first:])
first_pca = pd.concat(pca_chunks[:n_first])
second_pca = pd.concat(pca_chunks[n_first:])
print('=== Chronological Half-Split ===')
print(f'First half ({windows[0][2].date()} to {windows[n_first-1][3].date()}), n={n_first} windows:')
print(f'  SML-ERC-HRP Sharpe: {sharpe_lo2002(first_erc):.4f}')
print(f'  PCA-HRP     Sharpe: {sharpe_lo2002(first_pca):.4f}')
print(f'Second half ({windows[n_first][2].date()} to {windows[-1][3].date()}), n={len(windows)-n_first} windows:')
print(f'  SML-ERC-HRP Sharpe: {sharpe_lo2002(second_erc):.4f}')
print(f'  PCA-HRP     Sharpe: {sharpe_lo2002(second_pca):.4f}')
# Reconstruct SML-ERC allocations from fitted features; no optimiser is rerun.
# Model daily constant-mix rebalancing, consistent with the gross backtest.
# Cost rate applies per dollar bought or sold, includes initial entry from cash,
# excludes final liquidation, spreads beyond the assumed rate, taxes and market impact.
# Solve transaction costs against post-cost target holdings (self-financing).
erc_targets = []
for k, (ts, te, vs, ve) in enumerate(windows):
    train = returns.loc[ts:te]
    erc_targets.append(hrp_weights_from_dist(train, weighted_feature_distance(build_asset_features(train), erc_weights_log[k])))

def simulate_daily_costs(targets, rate):
    wealth = 1.0
    holdings = np.zeros(len(SYMBOLS))
    values, dates = [], []
    for target, (_, _, vs, ve) in zip(targets, windows):
        w = target.reindex(SYMBOLS).to_numpy()
        if not np.isfinite(w).all() or (w < 0).any() or not np.isclose(w.sum(), 1):
            raise ValueError('Invalid long-only target weights.')
        for date, row in returns.loc[vs:ve, SYMBOLS].iterrows():
            before = wealth
            lo, hi = 0.0, before
            for _ in range(60):
                after = (lo + hi) / 2
                balance = after + rate * np.abs(after*w - holdings).sum() - before
                if balance > 0: hi = after
                else: lo = after
            after = (lo + hi) / 2
            holdings = after*w*(1 + row.to_numpy())
            wealth = holdings.sum()
            values.append(wealth/before - 1)
            dates.append(date)
    return pd.Series(values, index=pd.DatetimeIndex(dates, name=returns.index.name))

cost_results = []
for name, targets, gross in [('SML-ERC', erc_targets, erc_series), ('PCA', pca_weights_per_window, pca_series)]:
    zero_cost = simulate_daily_costs(targets, 0.0)
    pd.testing.assert_index_equal(zero_cost.index, gross.index)
    np.testing.assert_allclose(zero_cost.values, gross.values, atol=1e-12, rtol=1e-9)
    for rate in (0.0, 0.0005, 0.0010):
        net = simulate_daily_costs(targets, rate)
        cost_results.append({'Method': name, 'Cost bps per dollar traded': rate*10000,
                             'Sharpe': sharpe_lo2002(net), 'MaxDD': max_drawdown(net),
                             'CEQ': ceq(net, 1), 'Terminal wealth': (1+net).prod()})
print(pd.DataFrame(cost_results).to_string(index=False))

# Live descriptive inputs only: a validated DSR requires a complete trial registry
# and consistent return frequency, moments, dependence treatment and selection model.
# Do not interpret these descriptive statistics as a selection-adjusted test.
dsr_inputs = {"n_daily": len(erc_series), "daily_mean": erc_series.mean(),
              "daily_std": erc_series.std(ddof=1), "daily_sharpe_unadjusted": erc_series.mean()/erc_series.std(ddof=1),
              "daily_skew": erc_series.skew(), "daily_kurtosis_ordinary": erc_series.kurt()+3,
              "pooled_adjusted_sharpe": sharpe_lo2002(erc_series)}
print(pd.Series(dsr_inputs))
print("DSR not reported: complete selection-trial registry and dependence specification required.")

# Current mean window Sharpes and differences; no historical constants.
for name, chunks in {"PCA": pca_chunks, "SML": sml_chunks, "Shrink": shrink_results[0.005]["chunks"], "ERC": erc_chunks, "CMA-ES": cmaes_chunks}.items():
    print(name, "mean window Sharpe", m(chunks), "difference versus PCA", m(chunks)-m(pca_chunks))


## 16. Deflated Sharpe ratio: declared-trial calculation

Calculate the Bailey–López de Prado DSR for SML-ERC from current gross daily returns, using ordinary daily Sharpe, daily skewness and ordinary kurtosis.


In [ ]:
# DSR for the explicitly listed Phase 2 trial family (gross, zero risk-free rate).
# Use DAILY ordinary Sharpe throughout, not the annualised Bartlett-adjusted ratio.
from scipy.stats import norm, skew, kurtosis

dsr_trials = {
    'Correlation': base_series, 'PCA': pca_series, 'AE': ae_series,
    'SML variance': sml_series, 'SML ERC': erc_series,
    'SML CMA-ES': cmaes_series, 'AE ERC': ae_erc_series,
    'Correlation custom': corr_herc_series, 'PCA custom': pca_herc_series,
    'SML custom': sml_herc_series, 'Regime shrinkage': rshrink_series,
}
# Include each nonzero shrinkage setting; lambda=0 repeats SML variance.
for lam, result in shrink_results.items():
    if float(lam) != 0:
        dsr_trials[f'Shrink lambda={lam}'] = result['series']

def dsr_statistics(candidate, trials, n_independent):
    if len(trials) < 2 or n_independent < 2:
        raise ValueError('At least two trials required.')
    for name, series in trials.items():
        if not candidate.index.equals(series.index):
            raise ValueError(f'Date mismatch: {name}')
        if len(series) < 3 or not np.isfinite(series.to_numpy()).all() or series.std(ddof=1) <= 0:
            raise ValueError(f'Invalid returns: {name}')
    sr_trials = np.array([s.mean()/s.std(ddof=1) for s in trials.values()])
    sr = float(candidate.mean()/candidate.std(ddof=1))
    # Across-trial dispersion, not the variance of quarterly Sharpe observations.
    sr_dispersion = float(sr_trials.std(ddof=1))
    euler_gamma = 0.5772156649015329
    sr0 = sr_dispersion * (
        (1-euler_gamma)*norm.ppf(1-1/n_independent)
        + euler_gamma*norm.ppf(1-1/(n_independent*np.e)))
    sk = float(skew(candidate.to_numpy(), bias=False))
    ku = float(kurtosis(candidate.to_numpy(), fisher=False, bias=False))
    variance_term = 1-sk*sr+(ku-1)*sr**2/4
    if not np.isfinite(variance_term) or variance_term <= 0:
        raise ValueError('Invalid Sharpe standard-error term.')
    se = np.sqrt(variance_term/(len(candidate)-1))
    z = (sr-sr0)/se
    return {'N assumed independent': n_independent, 'T daily': len(candidate),
            'Daily Sharpe': sr, 'Across-trial SR std': sr_dispersion,
            'Expected max daily SR': sr0, 'Skew': sk, 'Ordinary kurtosis': ku,
            'DSR (iid approximation)': norm.cdf(z),
            '1-DSR (stable tail)': norm.sf(z)}

print('Declared trial registry:', list(dsr_trials))
print('Candidate: SML ERC; gross returns; risk-free rate assumed zero.')
dsr_results = pd.DataFrame([
    dsr_statistics(erc_series, dsr_trials, count)
    for count in sorted(set([len(dsr_trials), 50, 100, 500]))
])
print(dsr_results.to_string(index=False, float_format=lambda x: f'{x:.8g}'))